# FLAG 2027 — NB-65x: B3-EVAL step 1b: the SAVED FACELORA-03 B3 adapter -> `clipB3` of every MAV-Celeb row (no training)

**What.** The CB 19.77 system (FACELORA-03 B3) scores faces with CLIP-L adapted by the B3 LoRA. On new test files (Evaluation) the
faces must go through the SAME adapter. This notebook loads the saved adapter (`facelora_state.pt` of
`quynhnhu82/flag2027-feats-ls-c5s-p`), skips training, and embeds every MAV-Celeb row found: v4 train, v1 / v2, and every
test list attached (dev and / or Evaluation; set `ONLY_UNDER` in the first cell when both are attached). Package: `BEST/v1977/`.

**Check:** the re-extracted arrays must equal the saved ones of the dataset (cosine >= 0.999 on every row of every array both have);
on Evaluation the new test lists have no reference, the train / v1 / v2 arrays still do.

**Inputs:** as FLAG_65 (`flag2027-raw-v4-p`, MAV-Celeb v1 / v2, `-ext-meta-p`, `-lab-data-p`, `-lab-vb-p`, `-feats-vox2b-p`,
`onnhqunhh/flag2027-lora-frames-p`) + **`quynhnhu82/flag2027-feats-ls-c5s-p`** (saved adapter + reference arrays) + on
11/11 the organisers' Evaluation data. **GPU T4** (CPU works, slower), **Internet ON** (CLIP-L). **Output:** `feats_clipb3_x/`
(`face_clipB3_<list>.npy`, `ext_*_face_clipB3.npy`) -> attach to FLAG_82b.

In [ ]:
print("[cell] 01 what to run (edit here)", flush=True)
print("[boot] FLAG_65xc_clipc5c_extract | python", __import__("sys").version.split()[0], "| start", __import__("time").strftime("%Y-%m-%d %H:%M:%S"), flush=True)  # heartbeat rule: first output, before any import
import os, sys, io, re, time, json, types, shutil, zipfile, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
LOCAL = os.environ.get("FLAG_LOCAL") == "1"          # CPU smoke test on a mini copy of the data
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
CACHE = Path(os.environ.get("FLAG_CACHE", "/tmp/flag_cache"))           # checkpoints stay out of the output
ECAPA_DIR = Path(os.environ.get("FLAG_ECAPA", str(CACHE / "ecapa_sb")))
OUT = WORKDIR / "feats_clipc5c_x"
for d in (WORKDIR, CACHE, OUT):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

# ------------------------------------------------------------------ what to run (edit here)
ENCODERS = ['clipC5V']   # CLIP-L adapted by B3-EVAL C5 (trained below)
EXT_SOURCES = ["v1_complete", "v2_complete"]                        # MAV-Celeb rows for STRESS-01; [] = skip
EXT_ENCODERS = list(ENCODERS)                                       # every model also on the v1/v2 rows
SPLIT_FILTER = None            # e.g. ["no_gender/English"]; None = every split found (train + every test list)
ZIP_PREFIXES = ("train_set", "dev_set", "eval", "test")             # organiser zips unpacked when not extracted
ONLY_UNDER = 'evaluation_phase'             # e.g. "eval": keep only lists whose path contains this (dev and eval both attached)
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| output", OUT, "| cache", CACHE)
SMOKE_ROWS = int(os.environ.get("FLAG_SMOKE_ROWS", "0"))   # local smoke test only: rows kept per split / source
if (LOCAL or os.environ.get("FLAG_SMOKE") == "1") and os.environ.get("FLAG_ENCODERS"):              # local smoke test only: a subset of the encoders
    ENCODERS = EXT_ENCODERS = os.environ["FLAG_ENCODERS"].split(",")
if (LOCAL or os.environ.get("FLAG_SMOKE") == "1") and os.environ.get("FLAG_SPLITS"):                # local smoke test only: a subset of the organiser lists
    SPLIT_FILTER = os.environ["FLAG_SPLITS"].split(",")
ARM, ARR, SG = "C5", "clipC5", 0.5   # B3-EVAL adapter of this notebook

In [ ]:
print("[cell] 02 openai CLIP (FaRLs code base, only for the T3 cells imports)", flush=True)
# openai CLIP (FaRL's code base, only for the T3 cell's imports) --no-deps; av is not needed (frames are cached)
if not LOCAL:
    !pip -q install ftfy regex 2>&1 | tail -1
    !pip -q install --no-deps "git+https://github.com/openai/CLIP.git@d50d76daa670286dd6cacf3bcd80b5e4823fc8e1" 2>&1 | tail -1
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, soundfile as sf, transformers
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEV,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

## 1. Find the data (FLAG_12 cell): organiser lists with media, v1/v2 rows, resume

In [ ]:
print("[cell] 03 finding the data", flush=True)
# ============================================================================ finding the data
def scan_inputs(inp: Path, media=("faces", "voices")):
    """One os.walk over the inputs that never enters a faces/ or voices/ folder (an extracted MAV-Celeb holds hundreds of
    thousands of files; every rglob over it costs minutes). The organiser csv live in features/{faces,voices}/: entered.
    -> {"files": name -> [paths], "roots": folders holding both faces/ and voices/}"""
    files, roots = {}, []
    for dp, dns, fns in os.walk(inp, followlinks=True):
        feat = Path(dp).name == "features"
        if not feat and all(m in dns for m in media):
            roots.append(Path(dp))
        dns[:] = sorted(d for d in dns if feat or d not in media)
        for f in fns:
            files.setdefault(f, []).append(Path(dp) / f)
    log(f"scanned {inp}: {sum(map(len, files.values()))} files outside media folders, {len(roots)} media roots")
    return dict(files=files, roots=roots)


def media_cols(txt: Path):
    """Column of the .wav and of the image in an organiser list (train: 2 / 3; test: 1 / 2), or None."""
    try:
        row = pd.read_csv(txt, sep=" ", header=None, nrows=1).iloc[0]
    except Exception:
        return None
    wav = [i for i, v in enumerate(row) if str(v).lower().endswith(".wav")]
    jpg = [i for i, v in enumerate(row) if str(v).lower().endswith((".jpg", ".jpeg", ".png"))]
    if not wav or not jpg:
        return None
    return wav[0], jpg[0]


def has_media(txt: Path, cols):
    row = pd.read_csv(txt, sep=" ", header=None, nrows=1).iloc[0]
    return (txt.parent / row[cols[0]]).exists() and (txt.parent / row[cols[1]]).exists()


def split_name(txt: Path):
    """train_English.txt -> 'train'; <folder>/English_test.txt -> '<folder>/English' (dev: 'no_gender/English')."""
    if txt.name == "train_English.txt":
        return "train"
    return f"{txt.parent.name}/{re.sub(r'_test$', '', txt.stem)}"


tag = lambda split: split.replace("/", "_")


def find_splits(scan):
    """Every organiser list whose media sit beside it. A features-only copy of a list (no media) is ignored.
    Two different lists with the same split name (dev and Evaluation both attached) stop the run: set ONLY_UNDER."""
    S = {}
    for name, paths in scan["files"].items():
        if not name.endswith(".txt") or not (name == "train_English.txt" or name.endswith("_test.txt")):
            continue
        for p in sorted(paths, key=lambda q: len(q.as_posix())):
            if ONLY_UNDER and ONLY_UNDER not in p.as_posix() and name != "train_English.txt":
                continue
            cols = media_cols(p)
            if cols is None or not has_media(p, cols):
                continue
            k = split_name(p)
            df = pd.read_csv(p, sep=" ", header=None)
            if k in S:
                same = len(df) == len(S[k]["wav"]) and df[cols[0]].tolist() == S[k]["wav"]
                assert same, f"two different lists named {k}: {S[k]['txt']} and {p} -> set ONLY_UNDER"
                continue
            S[k] = dict(txt=p, root=p.parent, wav=df[cols[0]].tolist(), jpg=df[cols[1]].tolist())
    return S


def unpack_zips(scan, work: Path):
    zips = [p for n, ps in scan["files"].items() if n.lower().endswith(".zip") and n.lower().startswith(ZIP_PREFIXES)
            for p in ps]
    for zp in zips:
        marker = work / f".{zp.stem}.done"
        if not marker.exists():
            log("unpacking", zp)
            with zipfile.ZipFile(zp) as z:
                z.extractall(work)
            marker.touch()
    return bool(zips)


SCAN = scan_inputs(IN)
S = find_splits(SCAN)
if "train" not in S or len(S) == 1:                  # something may still be zipped (Kaggle keeps some uploads as .zip)
    if unpack_zips(SCAN, WORKDIR / "raw"):
        for k, v in find_splits(scan_inputs(WORKDIR / "raw")).items():
            S.setdefault(k, v)
if SPLIT_FILTER:
    S = {k: v for k, v in S.items() if k in SPLIT_FILTER}
assert S, "no organiser list with media found: attach the raw v4 data (or the Evaluation data)"
def wav_duration(p):
    """Seconds; 1.0 for a file soundfile cannot open (load_audio then returns 1 s of silence for it)."""
    try:
        return sf.info(str(p)).duration
    except Exception:
        return 1.0


for k, v in S.items():
    v["dur"] = np.array([wav_duration(v["root"] / p) for p in v["wav"]], dtype=np.float32)
    shutil.copy(v["txt"], OUT / f"index_{tag(k)}.txt")
    log(f"{k:22s} {len(v['wav']):5d} rows | voice median {np.median(v['dur']):.1f}s, max {v['dur'].max():.1f}s | {v['txt']}")
meta = SCAN["files"].get("meta_file_train_set.csv")
if meta:
    shutil.copy(meta[0], OUT / "meta_file_train_set.csv")


# ============================================================================ MAV-Celeb v1/v2 rows (optional)
def media_key(member):
    """'.../faces/<id>/<lang>/<video>/<file>' -> 'faces/<id>/<lang>/<video>/<file>' (drops any version folder)."""
    p = member.replace("\\", "/").split("/")
    for kind in ("faces", "voices"):
        if kind in p:
            return "/".join(p[p.index(kind):])
    return None


def ext_source(scan, name):
    """The attached zip for `name` (largest copy), else the extracted folder whose path names the source."""
    norm = lambda t: t.lower().replace("-", "_")
    zips = [p for n, ps in scan["files"].items() if n.lower().endswith(".zip") and norm(name) in norm(n) for p in ps]
    if zips:
        return max(zips, key=lambda p: p.stat().st_size)
    short = name.split("_")[0]
    for r in scan["roots"]:
        hint = norm(r.as_posix()) + "/"
        if f"/{short}/" in hint or name in hint:
            return r
    return None


class ExtRows:
    """The exact rows of ext_<name>_{face,voice}_meta.csv, read from the attached zip or folder (row order = csv)."""

    def __init__(self, src, name, meta_dir: Path, max_sec=12.0):
        fm = pd.read_csv(meta_dir / f"ext_{name}_face_meta.csv").path
        vm = pd.read_csv(meta_dir / f"ext_{name}_voice_meta.csv").path
        src = Path(src)
        if src.is_dir():
            from concurrent.futures import ThreadPoolExecutor
            self.face, self.voice = [src / media_key(p) for p in fm], [src / media_key(p) for p in vm]
            with ThreadPoolExecutor(32) as ex:
                ok = list(ex.map(lambda q: q.exists(), self.face + self.voice))
            if not all(ok):
                raise KeyError(f"{ok.count(False)} rows missing, e.g. {(self.face + self.voice)[ok.index(False)]}")
            self.read = lambda q: q.read_bytes()
        else:
            z = zipfile.ZipFile(src)
            by = {media_key(i.filename): i.filename for i in z.infolist() if not i.is_dir() and media_key(i.filename)}
            self.face, self.voice = [by[media_key(p)] for p in fm], [by[media_key(p)] for p in vm]
            self.read = z.read
        self.L = int(max_sec * 16000)          # 12 s centre crop, as the organiser-space features of these rows
        log(f"{name}: {len(self.face)} face rows, {len(self.voice)} voice rows in {src}")


EXT = {}
for name in EXT_SOURCES:
    metas = sorted(SCAN["files"].get(f"ext_{name}_voice_meta.csv", []), key=lambda p: -p.stat().st_size)
    src = ext_source(SCAN, name)
    if metas and src is not None:
        try:
            EXT[name] = ExtRows(src, name, metas[0].parent)
        except Exception as e:                    # optional: a bad zip or a missing row must not stop the run
            log(f"{name}: skipped ({type(e).__name__}: {e})")
    else:
        log(f"{name}: skipped (meta csv found: {bool(metas)}, source found: {src is not None})")


# ============================================================================ resume
def seed_from_previous(scan, out: Path):
    """Copy arrays of an earlier run of THIS notebook (a folder named like flag2027-feats-eval / feats_eval), so finished
    work is skipped. Arrays of other datasets (feats_v2, feats_models) are references for the check, never copied."""
    copied = []
    for n, ps in scan["files"].items():
        if not n.endswith(".npy") or (out / n).exists():
            continue
        for p in ps:
            if re.search(r"feats[-_]clipc5c[-_]x", p.as_posix()) and out not in p.parents:
                shutil.copy(p, out / n); copied.append(n); break
    log(f"resume: {len(copied)} arrays copied from an earlier run")
    return copied


SEEDED = seed_from_previous(SCAN, OUT)
print({k: len(v["wav"]) for k, v in S.items()}, "| external:", list(EXT))
if (LOCAL or os.environ.get("FLAG_SMOKE") == "1") and SMOKE_ROWS:                     # local smoke test: the first rows of every split and source
    for v in S.values():
        v.update(wav=v["wav"][:SMOKE_ROWS], jpg=v["jpg"][:SMOKE_ROWS], dur=v["dur"][:SMOKE_ROWS])
    for r in EXT.values():
        r.face, r.voice = r.face[:SMOKE_ROWS], r.voice[:SMOKE_ROWS]
    log(f"smoke test: every split and source cut to {SMOKE_ROWS} rows")

## 2. Audio loading, batching, helpers (FLAG_12 cell)

In [ ]:
print("[cell] 04 audio + batching", flush=True)
# ============================================================================ audio + batching
TARGET_SR = 16000
AUDIO_STATS = {"files": 0, "resampled": 0, "multichannel": 0, "short": 0}
MIN_SEC = 0.1                  # shorter clips are tiled to 0.1 s (w2v2 / AudioCLIP STFT fail on a few ms; v4 clips are >= 2 s)
BROKEN = {"audio": [], "image": []}   # unreadable sources: scored from a stand-in, listed in the manifest


def _src_name(src):
    """Full path of a file source (base names repeat across splits); the archive member name of a zip stream."""
    if isinstance(src, (bytes, bytearray)):
        return f"<{len(src)} bytes>"
    return str(src) if isinstance(src, (str, Path)) else str(getattr(src, "name", None) or src)


def load_audio(src):
    """Mono float32 at 16 kHz as the organiser loads audio (librosa.load(sr=16000, mono=True): channel mean + soxr_hq).
    All v4 wavs are 16 kHz mono; MAV-Celeb v1 has 44.1 kHz files. `src` = path or file-like.
    A file that cannot be read (or is empty / non-finite) -> 1 s of silence, listed in BROKEN["audio"]."""
    try:
        w, sr = sf.read(src if hasattr(src, "read") else str(src), dtype="float32", always_2d=True)
        if w.size == 0 or not np.isfinite(w).all():
            raise ValueError(f"{w.shape[0]} samples, finite {bool(np.isfinite(w).all())}")
    except Exception as e:
        BROKEN["audio"].append(f"{_src_name(src)} | {type(e).__name__}: {e}")
        log("!! unreadable audio -> 1 s of silence:", BROKEN["audio"][-1])
        return np.zeros(TARGET_SR, np.float32)
    AUDIO_STATS["files"] += 1
    if w.shape[1] > 1:
        AUDIO_STATS["multichannel"] += 1
        w = w.mean(1, keepdims=True)
    w = w[:, 0]
    if sr != TARGET_SR:
        import librosa
        w = librosa.resample(w, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)
        AUDIO_STATS["resampled"] += 1
    if len(w) < int(MIN_SEC * TARGET_SR):
        AUDIO_STATS["short"] += 1
        w = np.tile(w, int(np.ceil(MIN_SEC * TARGET_SR / len(w))))[:int(MIN_SEC * TARGET_SR)]
    return w


def centre_crop(w, L):
    if len(w) <= L:
        return w
    s = (len(w) - L) // 2
    return w[s:s + L]


def pack_batches(durs, max_padded_sec=240.0, max_bs=32):
    """Longest first; a batch costs len(batch) * longest item after padding."""
    order = np.argsort(-np.asarray(durs), kind="stable")
    batches, cur = [], []
    for i in order:
        longest = durs[cur[0]] if cur else durs[i]
        if cur and ((len(cur) + 1) * longest > max_padded_sec or len(cur) >= max_bs):
            batches.append(cur); cur = []
        cur.append(int(i))
    if cur:
        batches.append(cur)
    return batches


def run_batched(items, fn, durs, max_padded_sec=240.0, max_bs=32, cpu_fn=None, label=""):
    """fn(list of items) -> list of outputs, in item order. A CUDA OOM halves the batch; a single item goes to CPU."""
    out = [None] * len(items)
    t0, done_n = time.time(), 0

    def go(idx):
        try:
            res = fn([items[i] for i in idx])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(idx) == 1:
                res = (cpu_fn or fn)([items[idx[0]]])
            else:
                h = len(idx) // 2
                go(idx[:h]); go(idx[h:])
                return
        for i, r in zip(idx, res):
            out[i] = r

    batches = pack_batches(durs, max_padded_sec, max_bs)
    for bi, b in enumerate(batches):
        go(b)
        done_n += len(b)
        if bi % 50 == 0 or done_n == len(items):
            el = time.time() - t0
            log(f"  {label} {done_n}/{len(items)}  {el:.0f}s  eta {el / done_n * (len(items) - done_n):.0f}s")
    return out


def cos_rows(a, b):
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    return np.sum(a * b, 1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-12)


def amp(flag):
    return torch.autocast("cuda", dtype=torch.float16, enabled=bool(flag) and DEV.type == "cuda")


def read_rgb(src):
    """PIL RGB image; an image that cannot be read -> a black 224 x 224 image, listed in BROKEN["image"]."""
    from PIL import Image
    try:
        return Image.open(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else src).convert("RGB")
    except Exception as e:
        BROKEN["image"].append(f"{_src_name(src)} | {type(e).__name__}: {e}")
        log("!! unreadable image -> black image:", BROKEN["image"][-1])
        return Image.new("RGB", (224, 224))

## 3. Track 3 encoder helpers (CLIP-L id)

In [ ]:
print("[cell] 05 T3ENC-01 encoders: the encoder family of Track 3s VoxBlink s", flush=True)
# ============================================================================ T3ENC-01 encoders: the encoder family of Track 3's VoxBlink system, frozen
# Track 3 (a teammate's VoxBlink pipeline, CB Bangla ng ~16 / g ~25) used these four encoders; here they run on OUR rows
# (MAV-Celeb v4 train, v1/v2, dev lists; and VoxCeleb2 in FLAG_41), so we can test locally whether the encoders and a
# cross-modal head carry the gain WITHOUT VoxBlink data (docs/PLAN_VOXBLINK_2026-10-07.md, V1). Every sample on its own.
#   farl16V   FaRL ViT-B/16 LAION-Face 20M **ep16** (github.com/FacePerceiver/FaRL release, 652 MB) on openai CLIP ViT-B/16
#             (CLIP preprocessing: 224 bicubic, centre crop); 512-d image embedding (our existing `farl` is the ep64 copy)
#   clipLV    openai/clip-vit-large-patch14 @ 32bd642 (transformers): get_image_features, 768-d
#   sam100pA  WeSpeaker voxblink2_samresnet100 (VoxBlink2 pretrained), official ModelScope zip (wenet/wespeaker_pretrained_models)
#   sam100fA  WeSpeaker voxblink2_samresnet100_ft (VoxBlink2 pretrained -> VoxCeleb2 large-margin fine-tune)
#             both: wespeaker fbank + CMN, int16-scaled waveform (as our wssim34 / wsr221 cells), 256-d
import zipfile as _zf

FARL16_URL = "https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep16.pth"
FARL16_SIZE = 652165542
CLIPL_ID, CLIPL_REV = "openai/clip-vit-large-patch14", "32bd64288804d66eefd0ccbe215aa642df71cc41"
WESPK_TREE = "https://modelscope.cn/api/v1/datasets/wenet/wespeaker_pretrained_models/oss/tree"
SAM_ZIPS = {"sam100pA": ("voxblink2_samresnet100.zip", 186890339), "sam100fA": ("voxblink2_samresnet100_ft.zip", 186890839)}
T3_STATS = {}


def fetch(url, dest, size=None):
    dest = Path(dest)
    if dest.exists() and (size is None or dest.stat().st_size == size):
        return dest
    torch.hub.download_url_to_file(url, str(dest) + ".part", progress=False)
    Path(str(dest) + ".part").replace(dest)
    assert size is None or dest.stat().st_size == size, f"{dest.name}: {dest.stat().st_size} bytes, expected {size}"
    return dest


class FaRL16Face:
    kind = "face"

    def __init__(self):
        import clip
        m, self.pre = clip.load("ViT-B/16", device="cpu", download_root=str(CACHE / "clip"))
        sd = torch.load(fetch(FARL16_URL, CACHE / "FaRL-ep16.pth", FARL16_SIZE), map_location="cpu", weights_only=False)["state_dict"]
        vis = [k for k in m.state_dict() if k.startswith("visual.")]
        hit = [k for k in vis if k in sd and sd[k].shape == m.state_dict()[k].shape]
        assert len(hit) == len(vis), f"FaRL ep16 covers {len(hit)}/{len(vis)} visual tensors"
        m.load_state_dict(sd, strict=False)
        self.m = m.visual.float().to(DEV).eval()

    def prep(self, im):
        return self.pre(im)

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        return {"": self.m(x.to(DEV)).float().cpu().numpy()}


class CLIPLFace:
    kind = "face"

    def __init__(self):
        from transformers import CLIPModel, CLIPImageProcessor
        self.proc = CLIPImageProcessor.from_pretrained(CLIPL_ID, revision=CLIPL_REV)
        self.m = CLIPModel.from_pretrained(CLIPL_ID, revision=CLIPL_REV, torch_dtype=torch.float32).to(DEV).eval()

    def prep(self, im):
        return self.proc(images=im.convert("RGB"), return_tensors="pt").pixel_values[0]

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        f = self.m.get_image_features(pixel_values=x.to(DEV))
        f = f.pooler_output if hasattr(f, "pooler_output") else f          # transformers 5 may return an output object
        return {"": f.float().cpu().numpy()}


def wespeaker_dir(zip_name, size):
    """The official zip (ModelScope, as wespeaker's own hub does), unpacked without the __MACOSX entries."""
    d = CACHE / "wespeaker" / zip_name[:-4]
    if (d / "avg_model.pt").exists() and (d / "config.yaml").exists():
        return d
    d.mkdir(parents=True, exist_ok=True)
    import requests
    tree = requests.get(WESPK_TREE, timeout=120).json()["Data"]
    url = [it["Url"] for it in tree if it.get("Key") == zip_name][0]
    z = fetch(url, d / zip_name, size)
    with _zf.ZipFile(z) as zf:
        for mem in zf.namelist():
            if not mem.startswith("__MACOSX") and not mem.endswith("/"):
                (d / Path(mem).name).write_bytes(zf.read(mem))
    # some releases (voxceleb_resnet293_LM) name the pair <model>.pt / <model>.yaml instead of avg_model.pt / config.yaml
    pts, ymls = sorted(d.glob("*.pt")), sorted(d.glob("*.yaml"))
    if not (d / "avg_model.pt").exists() and len(pts) == 1:
        pts[0].rename(d / "avg_model.pt")
    if not (d / "config.yaml").exists() and len(ymls) == 1:
        ymls[0].rename(d / "config.yaml")
    assert (d / "avg_model.pt").exists() and (d / "config.yaml").exists(), f"{zip_name}: unexpected content {zf.namelist()[:6]}"
    return d


class SimAMVoice:
    kind = "voice"

    def __init__(self, name):
        for mod, attr in (("wespeaker.frontend.s3prl", "S3prlFrontend"), ("wespeaker.frontend.whisper_encoder", "whisper_encoder"),
                          ("wespeaker.frontend.w2vbert", "W2VBertFrontend"), ("wespeaker.diar.umap_clusterer", "cluster"),
                          ("wespeaker.diar.extract_emb", "subsegment")):
            if mod not in sys.modules:                     # SSL frontends / diarisation helpers: not used by fbank models
                stub = types.ModuleType(mod); setattr(stub, attr, None); sys.modules[mod] = stub
        import wespeaker
        self.m = wespeaker.load_model(str(wespeaker_dir(*SAM_ZIPS[name])))
        self.m.set_device("cuda:0" if DEV.type == "cuda" else "cpu")
        T3_STATS[name] = {"params_M": round(sum(p.numel() for p in self.m.model.parameters()) / 1e6, 1)}

    @torch.no_grad()
    def __call__(self, w, use_amp=True):
        pcm = torch.from_numpy(np.ascontiguousarray(w, dtype=np.float32) * 32768.0)[None]
        feats = self.m.compute_features(pcm, sample_rate=16000, cmn=True)
        out = self.m.model(feats.to(self.m.device))
        out = out[-1] if isinstance(out, tuple) else out
        return {"": out[0].reshape(-1).float().cpu().numpy()}


NEW = {"farl16V": FaRL16Face, "clipLV": CLIPLFace, "sam100pA": lambda: SimAMVoice("sam100pA"),
       "sam100fA": lambda: SimAMVoice("sam100fA")}


def build(name):
    return NEW[name]()


def release(keep_ib=False):
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


ARRAYS_NEW = {"farl16V": ("face", {"": "farl16"}), "clipLV": ("face", {"": "clipL"}),
              "sam100pA": ("voice", {"": "sam100p"}), "sam100fA": ("voice", {"": "sam100f"})}
MAIN_NEW = {k: "" for k in NEW}
RESHAPE = {}

## 4. B3-EVAL: identities and frames of the three sources (registered filters)

In [ ]:
print("[cell] 06 FACELORA-03: data of the three sources (registered filters)", flush=True)
# ============================================================================ FACELORA-03: data of the three sources (registered filters)
import math, threading, requests
import concurrent.futures as cf
from torch.utils.checkpoint import checkpoint
from PIL import Image

SMOKE = (LOCAL or os.environ.get("FLAG_SMOKE") == "1")
CFG = dict(arm=ARM, enc="clipL", rank=16, alpha=32, blocks=list(range(12, 24)), attn=True, P=int(os.environ.get("LORA_P", "32")),
           K=int(os.environ.get("LORA_K", "4")), steps=int(os.environ.get("LORA_STEPS", "4600")), tau=0.05, sg=SG,
           lr_lora=1e-4, lr_proj=1e-3, wd=1e-2, warmup=200, held_out=300, seed=0, flip=0.5, ckpt_every=500)
SMOKE_IDS = int(os.environ.get("SMOKE_IDS", "0")) if SMOKE else 0
FILES = SCAN["files"]
l2n_ = lambda X: X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)


def one(name, where=None):
    c = [p for p in FILES.get(name, []) if where is None or where in p.as_posix()]
    assert c, f"{name} ({where}) not found in the inputs"
    return sorted(c, key=lambda p: len(p.as_posix()))[0]


def voices(d):
    return np.concatenate([l2n_(np.load(d / f"voice_{n}.npy").astype(np.float32)) for n in ("sam100p", "sam100f")], 1)


dt = {"spk": str, "video": str, "utt": str}
# -- VoxCeleb2 of FLAG_37 (lab-data: arrays in T3ENC-01/feats_vox2t3, key lists in VOX2-01/feats_vox2)
D37 = one("voice_sam100p.npy", "feats_vox2t3").parent
M37 = one("face_meta.csv", "VOX2-01/feats_vox2").parent
FK37, VK37 = pd.read_csv(M37 / "face_meta.csv", dtype=dt), pd.read_csv(M37 / "voice_meta.csv", dtype=dt)
VE37 = voices(D37)
# -- VoxCeleb2 of FLAG_53 (flag2027-feats-vox2b-p)
D2B = one("voice_sam100f.npy", "feats_vox2b").parent
FK2B, VK2B = pd.read_csv(D2B / "face_meta.csv", dtype=dt), pd.read_csv(D2B / "voice_meta.csv", dtype=dt)
VE2B = voices(D2B)
# -- VoxBlink-clean of FLAG_46 (flag2027-lab-vb-p)
DVB = one("voice_sam100f.npy", "SCALE-VB/feats_vb").parent
FKVB, VKVB = pd.read_csv(DVB / "face_meta.csv", dtype=dt), pd.read_csv(DVB / "voice_meta.csv", dtype=dt)
VEVB = voices(DVB)
assert len(VE37) == len(VK37) and len(VE2B) == len(VK2B) and len(VEVB) == len(VKVB)
# -- Bengali LID rules (speaker level) of FLAG_42, FLAG_53 (also drops old rows of its flagged speakers), FLAG_46
drop42 = set(json.loads(one("lid_summary.json", "feats_vox2lid").read_text())["speakers_dropped_ids"])
drop53 = set(json.loads((D2B / "lid_summary.json").read_text())["speakers_dropped_ids"])
dropvb = set(json.loads((DVB / "lid_summary.json").read_text())["speakers_dropped_ids"])
# -- VoxBlink noise rule of vb_rows: clip LOO cosine of SimAM ft below the 1st percentile of the same statistic on the
#    Bengali-filtered VoxCeleb2 clips of FLAG_37
def loo_cos(V, spk):
    """= SCALE-VB loo_cos: per clip, cosine to the mean of the OTHER clips of the same speaker (NaN for a single clip)."""
    out = np.full(len(V), np.nan)
    for _, idx in pd.Series(np.arange(len(V))).groupby(np.asarray(spk)).indices.items():
        if len(idx) < 2:
            continue
        S = V[idx].sum(0)
        out[idx] = np.einsum("ij,ij->i", V[idx], l2n_((S[None] - V[idx]) / (len(idx) - 1)))
    return out


XV = l2n_(np.load(D37 / "voice_sam100f.npy").astype(np.float32))
keepx = ~VK37.spk.isin(drop42).values
thr = float(np.nanquantile(loo_cos(XV[keepx], VK37.spk.values[keepx]), 0.01))
cvb = loo_cos(l2n_(np.load(DVB / "voice_sam100f.npy").astype(np.float32)), VKVB.spk.values)
vb_voice_ok = ~VKVB.spk.isin(dropvb).values & ~(cvb < thr)
log(f"VoxBlink noise threshold {thr:.4f}: {int((~vb_voice_ok).sum())} voice clips dropped (LID + noise)")
# -- genders: FLAG_37 face_meta; VoxCeleb2 official meta (gender column only) for FLAG_53; VoxBlink face_meta
VOX2_GENDER_JSON = r"""{
"id00012": "m",
"id00015": "m",
"id00016": "m",
"id00017": "m",
"id00018": "m",
"id00019": "m",
"id00020": "m",
"id00021": "m",
"id00022": "f",
"id00024": "f",
"id00025": "f",
"id00026": "m",
"id00027": "m",
"id00028": "m",
"id00029": "m",
"id00032": "m",
"id00033": "m",
"id00035": "m",
"id00036": "m",
"id00039": "f",
"id00040": "m",
"id00042": "m",
"id00043": "f",
"id00044": "m",
"id00046": "m",
"id00047": "m",
"id00049": "m",
"id00050": "m",
"id00051": "m",
"id00052": "m",
"id00053": "m",
"id00055": "m",
"id00056": "m",
"id00058": "m",
"id00059": "m",
"id00060": "m",
"id00061": "m",
"id00062": "m",
"id00063": "m",
"id00064": "m",
"id00066": "m",
"id00067": "f",
"id00068": "f",
"id00069": "f",
"id00070": "m",
"id00071": "f",
"id00073": "f",
"id00074": "m",
"id00075": "m",
"id00076": "f",
"id00078": "m",
"id00079": "m",
"id00080": "m",
"id00081": "m",
"id00082": "m",
"id00083": "m",
"id00084": "m",
"id00085": "m",
"id00086": "m",
"id00087": "m",
"id00088": "m",
"id00089": "m",
"id00090": "f",
"id00091": "f",
"id00092": "f",
"id00096": "m",
"id00097": "f",
"id00098": "f",
"id00099": "f",
"id00100": "f",
"id00103": "m",
"id00104": "f",
"id00105": "f",
"id00109": "f",
"id00111": "f",
"id00114": "f",
"id00116": "m",
"id00117": "m",
"id00126": "m",
"id00127": "m",
"id00128": "m",
"id00129": "m",
"id00134": "f",
"id00137": "f",
"id00144": "f",
"id00145": "f",
"id00146": "f",
"id00149": "f",
"id00151": "m",
"id00154": "m",
"id00155": "f",
"id00159": "m",
"id00161": "m",
"id00163": "m",
"id00164": "m",
"id00165": "m",
"id00166": "m",
"id00167": "m",
"id00168": "m",
"id00169": "m",
"id00170": "m",
"id00171": "m",
"id00172": "m",
"id00173": "m",
"id00175": "m",
"id00176": "m",
"id00177": "m",
"id00179": "m",
"id00180": "f",
"id00181": "m",
"id00183": "m",
"id00184": "m",
"id00185": "m",
"id00186": "m",
"id00187": "m",
"id00188": "f",
"id00189": "f",
"id00190": "f",
"id00191": "m",
"id00192": "m",
"id00195": "m",
"id00197": "m",
"id00198": "m",
"id00201": "m",
"id00202": "m",
"id00203": "m",
"id00206": "m",
"id00220": "m",
"id00221": "m",
"id00222": "m",
"id00223": "m",
"id00225": "m",
"id00231": "f",
"id00232": "f",
"id00234": "f",
"id00235": "f",
"id00236": "f",
"id00238": "f",
"id00239": "f",
"id00241": "m",
"id00242": "m",
"id00243": "m",
"id00244": "m",
"id00245": "m",
"id00246": "m",
"id00249": "m",
"id00251": "f",
"id00253": "m",
"id00254": "m",
"id00255": "m",
"id00257": "m",
"id00258": "m",
"id00259": "f",
"id00261": "f",
"id00262": "f",
"id00264": "m",
"id00265": "m",
"id00266": "m",
"id00268": "m",
"id00269": "m",
"id00270": "f",
"id00271": "f",
"id00272": "f",
"id00274": "m",
"id00275": "m",
"id00276": "m",
"id00277": "m",
"id00279": "m",
"id00281": "m",
"id00282": "m",
"id00283": "m",
"id00285": "f",
"id00286": "f",
"id00287": "f",
"id00288": "f",
"id00289": "f",
"id00290": "f",
"id00291": "f",
"id00292": "m",
"id00296": "m",
"id00297": "f",
"id00298": "m",
"id00299": "f",
"id00300": "m",
"id00301": "f",
"id00302": "m",
"id00303": "m",
"id00304": "m",
"id00305": "m",
"id00306": "m",
"id00309": "m",
"id00311": "m",
"id00312": "m",
"id00313": "m",
"id00314": "m",
"id00316": "m",
"id00317": "m",
"id00318": "m",
"id00319": "f",
"id00320": "f",
"id00322": "m",
"id00323": "f",
"id00324": "f",
"id00325": "f",
"id00328": "f",
"id00329": "f",
"id00330": "f",
"id00331": "f",
"id00332": "f",
"id00336": "f",
"id00341": "f",
"id00342": "f",
"id00343": "f",
"id00344": "f",
"id00345": "m",
"id00346": "f",
"id00347": "f",
"id00348": "m",
"id00350": "m",
"id00351": "f",
"id00352": "m",
"id00353": "m",
"id00354": "f",
"id00355": "f",
"id00356": "f",
"id00357": "f",
"id00358": "m",
"id00359": "f",
"id00361": "m",
"id00363": "f",
"id00365": "f",
"id00366": "m",
"id00368": "m",
"id00371": "f",
"id00372": "f",
"id00373": "f",
"id00374": "f",
"id00376": "f",
"id00377": "f",
"id00378": "f",
"id00379": "f",
"id00380": "f",
"id00381": "f",
"id00383": "f",
"id00385": "f",
"id00386": "f",
"id00387": "f",
"id00388": "f",
"id00389": "m",
"id00390": "m",
"id00391": "m",
"id00393": "m",
"id00395": "f",
"id00397": "f",
"id00398": "f",
"id00399": "f",
"id00401": "f",
"id00402": "f",
"id00403": "f",
"id00405": "f",
"id00406": "f",
"id00407": "m",
"id00409": "m",
"id00411": "f",
"id00412": "m",
"id00413": "m",
"id00414": "m",
"id00415": "m",
"id00416": "m",
"id00417": "f",
"id00418": "f",
"id00419": "f",
"id00420": "f",
"id00421": "f",
"id00422": "f",
"id00423": "f",
"id00425": "f",
"id00426": "f",
"id00427": "f",
"id00428": "f",
"id00429": "f",
"id00430": "f",
"id00431": "f",
"id00432": "f",
"id00434": "f",
"id00435": "f",
"id00436": "m",
"id00437": "f",
"id00438": "f",
"id00444": "f",
"id00445": "f",
"id00448": "f",
"id00449": "f",
"id00450": "f",
"id00456": "f",
"id00457": "f",
"id00458": "f",
"id00459": "m",
"id00460": "f",
"id00461": "f",
"id00462": "f",
"id00463": "f",
"id00464": "f",
"id00465": "f",
"id00467": "m",
"id00468": "m",
"id00469": "m",
"id00472": "f",
"id00473": "f",
"id00475": "m",
"id00476": "m",
"id00478": "m",
"id00479": "m",
"id00481": "m",
"id00482": "f",
"id00483": "m",
"id00484": "f",
"id00486": "f",
"id00488": "f",
"id00489": "f",
"id00490": "m",
"id00491": "f",
"id00492": "f",
"id00494": "f",
"id00495": "f",
"id00496": "f",
"id00498": "m",
"id00500": "f",
"id00501": "f",
"id00502": "f",
"id00503": "f",
"id00504": "f",
"id00505": "f",
"id00506": "m",
"id00507": "m",
"id00508": "m",
"id00509": "m",
"id00510": "m",
"id00512": "m",
"id00513": "m",
"id00514": "m",
"id00515": "f",
"id00516": "m",
"id00517": "m",
"id00518": "m",
"id00519": "m",
"id00520": "m",
"id00521": "m",
"id00522": "m",
"id00524": "m",
"id00525": "m",
"id00526": "m",
"id00527": "m",
"id00528": "m",
"id00529": "m",
"id00530": "m",
"id00531": "m",
"id00534": "m",
"id00535": "m",
"id00539": "m",
"id00542": "m",
"id00543": "m",
"id00544": "m",
"id00545": "m",
"id00546": "m",
"id00548": "m",
"id00550": "m",
"id00552": "m",
"id00553": "m",
"id00554": "m",
"id00555": "m",
"id00557": "m",
"id00558": "m",
"id00559": "m",
"id00560": "m",
"id00561": "m",
"id00562": "m",
"id00566": "f",
"id00567": "f",
"id00568": "f",
"id00569": "f",
"id00570": "f",
"id00571": "f",
"id00573": "f",
"id00574": "f",
"id00575": "f",
"id00577": "f",
"id00578": "f",
"id00579": "f",
"id00581": "f",
"id00582": "f",
"id00584": "f",
"id00585": "f",
"id00586": "m",
"id00587": "m",
"id00588": "f",
"id00589": "f",
"id00591": "f",
"id00592": "f",
"id00593": "f",
"id00594": "m",
"id00596": "m",
"id00597": "m",
"id00607": "f",
"id00608": "f",
"id00610": "f",
"id00611": "f",
"id00612": "f",
"id00613": "f",
"id00614": "f",
"id00615": "f",
"id00616": "f",
"id00617": "f",
"id00618": "f",
"id00620": "f",
"id00621": "f",
"id00622": "f",
"id00623": "f",
"id00624": "f",
"id00628": "f",
"id00629": "f",
"id00631": "f",
"id00632": "f",
"id00633": "f",
"id00634": "f",
"id00636": "f",
"id00637": "f",
"id00638": "f",
"id00641": "f",
"id00642": "f",
"id00643": "f",
"id00644": "f",
"id00645": "f",
"id00647": "f",
"id00648": "f",
"id00649": "f",
"id00650": "f",
"id00651": "f",
"id00652": "f",
"id00653": "f",
"id00654": "f",
"id00655": "f",
"id00656": "f",
"id00657": "f",
"id00659": "f",
"id00663": "f",
"id00664": "f",
"id00665": "f",
"id00666": "f",
"id00667": "f",
"id00668": "f",
"id00669": "f",
"id00670": "f",
"id00672": "f",
"id00673": "f",
"id00674": "f",
"id00675": "f",
"id00676": "f",
"id00677": "f",
"id00678": "f",
"id00680": "f",
"id00681": "f",
"id00685": "m",
"id00686": "m",
"id00687": "f",
"id00690": "m",
"id00691": "m",
"id00692": "m",
"id00693": "m",
"id00694": "m",
"id00695": "m",
"id00696": "m",
"id00699": "m",
"id00700": "f",
"id00701": "m",
"id00707": "f",
"id00708": "m",
"id00709": "m",
"id00710": "m",
"id00711": "m",
"id00712": "m",
"id00714": "m",
"id00715": "m",
"id00717": "m",
"id00721": "m",
"id00723": "m",
"id00724": "m",
"id00725": "m",
"id00726": "f",
"id00727": "m",
"id00729": "m",
"id00730": "f",
"id00731": "m",
"id00732": "m",
"id00735": "f",
"id00736": "m",
"id00739": "f",
"id00740": "m",
"id00741": "m",
"id00742": "m",
"id00745": "m",
"id00746": "f",
"id00747": "f",
"id00750": "f",
"id00752": "f",
"id00753": "f",
"id00754": "f",
"id00755": "f",
"id00758": "f",
"id00759": "f",
"id00760": "m",
"id00761": "m",
"id00762": "m",
"id00763": "f",
"id00765": "m",
"id00766": "m",
"id00769": "m",
"id00770": "f",
"id00773": "m",
"id00774": "m",
"id00775": "m",
"id00776": "m",
"id00777": "m",
"id00778": "m",
"id00779": "m",
"id00780": "m",
"id00781": "m",
"id00782": "m",
"id00783": "m",
"id00784": "m",
"id00785": "m",
"id00786": "m",
"id00787": "m",
"id00788": "m",
"id00792": "m",
"id00794": "m",
"id00796": "m",
"id00797": "m",
"id00799": "m",
"id00801": "m",
"id00802": "m",
"id00803": "f",
"id00804": "f",
"id00805": "m",
"id00806": "f",
"id00809": "f",
"id00810": "f",
"id00811": "f",
"id00812": "f",
"id00813": "f",
"id00814": "m",
"id00815": "m",
"id00816": "m",
"id00817": "m",
"id00818": "f",
"id00820": "f",
"id00823": "f",
"id00824": "f",
"id00825": "f",
"id00826": "f",
"id00827": "m",
"id00828": "f",
"id00829": "f",
"id00830": "m",
"id00831": "m",
"id00832": "m",
"id00833": "f",
"id00835": "f",
"id00841": "f",
"id00842": "f",
"id00843": "f",
"id00844": "f",
"id00846": "f",
"id00848": "f",
"id00851": "f",
"id00852": "m",
"id00857": "m",
"id00859": "f",
"id00860": "m",
"id00863": "m",
"id00865": "f",
"id00866": "m",
"id00867": "m",
"id00868": "m",
"id00869": "m",
"id00875": "m",
"id00876": "f",
"id00879": "m",
"id00880": "m",
"id00881": "m",
"id00882": "m",
"id00883": "m",
"id00884": "m",
"id00885": "f",
"id00886": "f",
"id00888": "f",
"id00890": "f",
"id00891": "f",
"id00892": "f",
"id00893": "f",
"id00894": "f",
"id00895": "f",
"id00897": "f",
"id00898": "f",
"id00899": "f",
"id00901": "f",
"id00902": "f",
"id00903": "m",
"id00904": "m",
"id00905": "m",
"id00906": "m",
"id00907": "m",
"id00908": "m",
"id00909": "m",
"id00910": "m",
"id00911": "m",
"id00917": "m",
"id00918": "m",
"id00919": "m",
"id00920": "f",
"id00922": "f",
"id00923": "f",
"id00924": "f",
"id00925": "f",
"id00926": "m",
"id00927": "m",
"id00928": "m",
"id00929": "f",
"id00930": "f",
"id00931": "m",
"id00933": "f",
"id00934": "f",
"id00935": "f",
"id00937": "f",
"id00938": "f",
"id00939": "f",
"id00941": "f",
"id00942": "m",
"id00943": "m",
"id00944": "m",
"id00945": "m",
"id00946": "m",
"id00947": "m",
"id00948": "m",
"id00959": "f",
"id00960": "m",
"id00963": "m",
"id00964": "m",
"id00965": "m",
"id00966": "m",
"id00967": "m",
"id00969": "m",
"id00971": "m",
"id00974": "f",
"id00975": "m",
"id00976": "m",
"id00978": "m",
"id00979": "m",
"id00980": "m",
"id00981": "m",
"id00982": "m",
"id00983": "m",
"id00984": "m",
"id00985": "m",
"id00987": "m",
"id00989": "m",
"id00990": "m",
"id00991": "m",
"id00992": "m",
"id00993": "m",
"id00994": "m",
"id00995": "m",
"id00996": "m",
"id00997": "m",
"id00999": "m",
"id01000": "f",
"id01001": "f",
"id01002": "f",
"id01003": "f",
"id01004": "f",
"id01005": "f",
"id01006": "f",
"id01007": "m",
"id01008": "f",
"id01010": "f",
"id01011": "f",
"id01014": "f",
"id01015": "f",
"id01016": "f",
"id01017": "f",
"id01018": "f",
"id01021": "f",
"id01023": "m",
"id01024": "f",
"id01025": "f",
"id01026": "f",
"id01028": "f",
"id01029": "f",
"id01030": "f",
"id01031": "f",
"id01035": "m",
"id01036": "m",
"id01037": "f",
"id01038": "m",
"id01039": "f",
"id01040": "m",
"id01041": "m",
"id01042": "m",
"id01043": "m",
"id01044": "m",
"id01045": "m",
"id01047": "m",
"id01048": "m",
"id01049": "m",
"id01050": "m",
"id01051": "m",
"id01052": "m",
"id01053": "m",
"id01054": "m",
"id01056": "m",
"id01057": "m",
"id01058": "m",
"id01060": "m",
"id01061": "m",
"id01062": "m",
"id01063": "m",
"id01065": "m",
"id01066": "m",
"id01067": "m",
"id01068": "m",
"id01069": "f",
"id01070": "m",
"id01071": "f",
"id01073": "f",
"id01074": "f",
"id01075": "f",
"id01076": "m",
"id01078": "m",
"id01080": "m",
"id01081": "m",
"id01083": "m",
"id01084": "m",
"id01085": "f",
"id01090": "f",
"id01091": "f",
"id01092": "m",
"id01093": "m",
"id01094": "f",
"id01095": "m",
"id01096": "m",
"id01097": "m",
"id01098": "m",
"id01099": "m",
"id01100": "m",
"id01101": "m",
"id01102": "m",
"id01103": "m",
"id01104": "m",
"id01105": "m",
"id01106": "m",
"id01108": "m",
"id01109": "m",
"id01110": "m",
"id01111": "m",
"id01112": "m",
"id01113": "m",
"id01115": "m",
"id01116": "f",
"id01117": "f",
"id01119": "m",
"id01120": "m",
"id01121": "m",
"id01122": "m",
"id01123": "m",
"id01124": "m",
"id01125": "m",
"id01126": "m",
"id01129": "m",
"id01130": "m",
"id01132": "m",
"id01139": "f",
"id01140": "f",
"id01141": "m",
"id01142": "m",
"id01149": "m",
"id01150": "m",
"id01154": "m",
"id01155": "m",
"id01156": "m",
"id01157": "m",
"id01158": "m",
"id01159": "m",
"id01160": "m",
"id01161": "m",
"id01162": "m",
"id01163": "m",
"id01164": "m",
"id01165": "m",
"id01166": "m",
"id01167": "m",
"id01168": "m",
"id01169": "m",
"id01170": "m",
"id01171": "m",
"id01172": "m",
"id01173": "m",
"id01174": "m",
"id01175": "m",
"id01178": "f",
"id01179": "m",
"id01182": "m",
"id01183": "m",
"id01184": "f",
"id01185": "m",
"id01187": "f",
"id01188": "m",
"id01189": "m",
"id01190": "m",
"id01191": "m",
"id01192": "m",
"id01193": "m",
"id01196": "m",
"id01199": "m",
"id01200": "m",
"id01201": "m",
"id01203": "m",
"id01204": "m",
"id01205": "m",
"id01206": "m",
"id01207": "m",
"id01209": "m",
"id01210": "m",
"id01211": "m",
"id01212": "m",
"id01213": "m",
"id01214": "m",
"id01215": "m",
"id01216": "f",
"id01217": "f",
"id01218": "m",
"id01220": "f",
"id01221": "f",
"id01222": "f",
"id01223": "f",
"id01224": "f",
"id01225": "f",
"id01227": "f",
"id01228": "f",
"id01229": "f",
"id01230": "f",
"id01231": "f",
"id01232": "f",
"id01234": "f",
"id01235": "f",
"id01236": "f",
"id01237": "f",
"id01238": "f",
"id01239": "m",
"id01240": "m",
"id01241": "m",
"id01243": "m",
"id01244": "m",
"id01245": "f",
"id01246": "f",
"id01248": "f",
"id01249": "f",
"id01251": "f",
"id01252": "m",
"id01253": "m",
"id01254": "m",
"id01255": "m",
"id01256": "m",
"id01257": "m",
"id01258": "f",
"id01261": "m",
"id01262": "m",
"id01263": "m",
"id01264": "m",
"id01265": "m",
"id01266": "m",
"id01268": "m",
"id01269": "m",
"id01270": "m",
"id01271": "m",
"id01272": "m",
"id01273": "m",
"id01274": "m",
"id01275": "m",
"id01276": "m",
"id01279": "m",
"id01280": "m",
"id01281": "f",
"id01284": "m",
"id01285": "f",
"id01286": "m",
"id01288": "m",
"id01289": "f",
"id01290": "f",
"id01291": "f",
"id01292": "f",
"id01293": "m",
"id01294": "m",
"id01295": "f",
"id01296": "m",
"id01298": "m",
"id01300": "m",
"id01301": "m",
"id01303": "f",
"id01304": "f",
"id01305": "f",
"id01306": "f",
"id01309": "f",
"id01313": "m",
"id01314": "f",
"id01315": "m",
"id01316": "f",
"id01317": "f",
"id01323": "f",
"id01325": "m",
"id01326": "f",
"id01327": "m",
"id01328": "m",
"id01329": "f",
"id01330": "f",
"id01331": "f",
"id01332": "f",
"id01333": "f",
"id01334": "f",
"id01335": "f",
"id01337": "f",
"id01338": "f",
"id01339": "f",
"id01340": "f",
"id01341": "f",
"id01342": "f",
"id01344": "m",
"id01346": "f",
"id01347": "f",
"id01348": "f",
"id01349": "f",
"id01350": "f",
"id01351": "f",
"id01352": "m",
"id01353": "m",
"id01354": "m",
"id01355": "m",
"id01356": "m",
"id01358": "f",
"id01359": "f",
"id01361": "f",
"id01362": "f",
"id01363": "m",
"id01364": "f",
"id01365": "f",
"id01367": "m",
"id01368": "m",
"id01369": "m",
"id01370": "m",
"id01371": "m",
"id01372": "m",
"id01373": "m",
"id01374": "m",
"id01376": "m",
"id01379": "m",
"id01380": "m",
"id01384": "m",
"id01385": "f",
"id01386": "m",
"id01387": "f",
"id01388": "f",
"id01389": "f",
"id01391": "f",
"id01392": "m",
"id01394": "f",
"id01395": "f",
"id01396": "f",
"id01397": "f",
"id01399": "f",
"id01402": "m",
"id01403": "f",
"id01404": "f",
"id01405": "f",
"id01406": "f",
"id01407": "f",
"id01408": "f",
"id01409": "f",
"id01410": "f",
"id01411": "f",
"id01413": "f",
"id01418": "f",
"id01420": "f",
"id01422": "f",
"id01423": "f",
"id01424": "f",
"id01425": "f",
"id01426": "f",
"id01429": "f",
"id01430": "f",
"id01431": "f",
"id01432": "f",
"id01433": "f",
"id01434": "f",
"id01436": "f",
"id01437": "m",
"id01438": "m",
"id01439": "m",
"id01445": "f",
"id01447": "m",
"id01448": "f",
"id01449": "f",
"id01450": "f",
"id01451": "f",
"id01452": "f",
"id01455": "f",
"id01456": "f",
"id01457": "f",
"id01458": "f",
"id01459": "f",
"id01460": "f",
"id01461": "f",
"id01464": "f",
"id01466": "f",
"id01468": "f",
"id01470": "m",
"id01471": "m",
"id01476": "f",
"id01487": "m",
"id01488": "f",
"id01489": "m",
"id01490": "m",
"id01491": "m",
"id01492": "m",
"id01493": "m",
"id01497": "m",
"id01498": "m",
"id01499": "m",
"id01500": "m",
"id01501": "m",
"id01502": "f",
"id01503": "m",
"id01504": "f",
"id01505": "m",
"id01508": "f",
"id01509": "m",
"id01510": "f",
"id01511": "f",
"id01512": "f",
"id01513": "f",
"id01514": "f",
"id01515": "f",
"id01516": "f",
"id01521": "m",
"id01522": "m",
"id01523": "m",
"id01524": "m",
"id01527": "m",
"id01528": "m",
"id01529": "m",
"id01530": "m",
"id01531": "f",
"id01532": "f",
"id01533": "m",
"id01534": "m",
"id01535": "m",
"id01536": "m",
"id01539": "m",
"id01540": "m",
"id01541": "m",
"id01542": "m",
"id01543": "m",
"id01544": "m",
"id01545": "m",
"id01546": "m",
"id01547": "f",
"id01548": "f",
"id01549": "f",
"id01550": "f",
"id01551": "f",
"id01552": "f",
"id01554": "f",
"id01555": "f",
"id01556": "f",
"id01557": "m",
"id01558": "m",
"id01560": "f",
"id01561": "f",
"id01564": "m",
"id01566": "f",
"id01567": "f",
"id01568": "f",
"id01569": "f",
"id01570": "m",
"id01571": "m",
"id01572": "f",
"id01574": "f",
"id01578": "f",
"id01579": "m",
"id01580": "m",
"id01581": "f",
"id01582": "f",
"id01583": "f",
"id01584": "f",
"id01585": "f",
"id01589": "m",
"id01590": "m",
"id01591": "m",
"id01592": "m",
"id01593": "m",
"id01594": "m",
"id01595": "m",
"id01596": "m",
"id01597": "m",
"id01598": "m",
"id01599": "m",
"id01605": "m",
"id01608": "m",
"id01609": "m",
"id01610": "m",
"id01611": "m",
"id01613": "m",
"id01614": "m",
"id01615": "f",
"id01617": "f",
"id01618": "f",
"id01619": "f",
"id01620": "f",
"id01624": "m",
"id01625": "m",
"id01626": "m",
"id01627": "m",
"id01628": "m",
"id01629": "m",
"id01630": "m",
"id01633": "m",
"id01634": "m",
"id01637": "m",
"id01639": "m",
"id01640": "m",
"id01641": "m",
"id01643": "m",
"id01646": "f",
"id01648": "f",
"id01649": "f",
"id01650": "f",
"id01652": "f",
"id01653": "f",
"id01654": "f",
"id01655": "f",
"id01656": "f",
"id01657": "f",
"id01658": "f",
"id01659": "f",
"id01660": "f",
"id01661": "f",
"id01662": "f",
"id01663": "m",
"id01664": "m",
"id01666": "m",
"id01667": "m",
"id01668": "m",
"id01669": "m",
"id01671": "m",
"id01672": "m",
"id01674": "m",
"id01676": "m",
"id01677": "m",
"id01678": "m",
"id01680": "m",
"id01681": "f",
"id01682": "f",
"id01683": "m",
"id01684": "m",
"id01685": "m",
"id01686": "m",
"id01687": "m",
"id01688": "m",
"id01689": "m",
"id01690": "m",
"id01691": "m",
"id01692": "f",
"id01693": "m",
"id01694": "f",
"id01695": "m",
"id01696": "f",
"id01697": "f",
"id01699": "m",
"id01708": "f",
"id01710": "f",
"id01711": "f",
"id01713": "m",
"id01716": "m",
"id01717": "m",
"id01718": "m",
"id01719": "f",
"id01720": "f",
"id01721": "f",
"id01722": "f",
"id01723": "f",
"id01725": "f",
"id01726": "f",
"id01727": "f",
"id01728": "f",
"id01729": "f",
"id01733": "m",
"id01734": "m",
"id01735": "m",
"id01736": "m",
"id01738": "m",
"id01739": "m",
"id01741": "m",
"id01742": "m",
"id01743": "m",
"id01745": "m",
"id01746": "m",
"id01748": "m",
"id01749": "m",
"id01750": "m",
"id01751": "m",
"id01753": "m",
"id01754": "m",
"id01755": "m",
"id01756": "f",
"id01758": "m",
"id01759": "m",
"id01760": "f",
"id01761": "f",
"id01762": "m",
"id01763": "m",
"id01764": "f",
"id01765": "f",
"id01766": "f",
"id01767": "m",
"id01768": "m",
"id01769": "m",
"id01770": "m",
"id01772": "m",
"id01773": "f",
"id01774": "f",
"id01775": "f",
"id01776": "m",
"id01777": "m",
"id01778": "m",
"id01779": "m",
"id01780": "m",
"id01781": "m",
"id01782": "m",
"id01783": "f",
"id01785": "f",
"id01786": "m",
"id01787": "m",
"id01788": "m",
"id01792": "f",
"id01793": "m",
"id01794": "f",
"id01795": "m",
"id01796": "f",
"id01797": "f",
"id01798": "m",
"id01799": "m",
"id01800": "f",
"id01801": "m",
"id01802": "f",
"id01803": "m",
"id01805": "m",
"id01806": "m",
"id01808": "m",
"id01809": "m",
"id01821": "m",
"id01822": "m",
"id01823": "f",
"id01826": "f",
"id01827": "f",
"id01829": "m",
"id01830": "m",
"id01832": "f",
"id01833": "f",
"id01835": "m",
"id01836": "m",
"id01837": "f",
"id01838": "f",
"id01840": "f",
"id01841": "f",
"id01842": "m",
"id01848": "m",
"id01849": "m",
"id01852": "m",
"id01853": "m",
"id01854": "m",
"id01856": "m",
"id01857": "m",
"id01858": "m",
"id01859": "m",
"id01861": "m",
"id01862": "m",
"id01864": "f",
"id01866": "f",
"id01868": "m",
"id01869": "f",
"id01870": "f",
"id01873": "f",
"id01874": "f",
"id01875": "m",
"id01876": "m",
"id01877": "m",
"id01878": "m",
"id01879": "m",
"id01880": "m",
"id01881": "f",
"id01883": "m",
"id01885": "m",
"id01886": "m",
"id01887": "m",
"id01888": "m",
"id01889": "m",
"id01890": "m",
"id01891": "m",
"id01892": "m",
"id01893": "m",
"id01894": "m",
"id01895": "m",
"id01898": "m",
"id01899": "m",
"id01900": "m",
"id01904": "m",
"id01905": "f",
"id01906": "m",
"id01907": "m",
"id01908": "m",
"id01909": "f",
"id01910": "m",
"id01912": "m",
"id01913": "m",
"id01915": "m",
"id01916": "m",
"id01918": "m",
"id01919": "m",
"id01920": "m",
"id01921": "m",
"id01925": "m",
"id01927": "m",
"id01928": "m",
"id01931": "m",
"id01932": "m",
"id01933": "m",
"id01934": "m",
"id01937": "f",
"id01939": "f",
"id01941": "f",
"id01945": "m",
"id01946": "f",
"id01947": "f",
"id01948": "f",
"id01949": "f",
"id01951": "f",
"id01953": "m",
"id01955": "m",
"id01959": "f",
"id01960": "m",
"id01961": "f",
"id01962": "f",
"id01963": "m",
"id01964": "m",
"id01965": "m",
"id01966": "m",
"id01967": "m",
"id01969": "m",
"id01971": "m",
"id01972": "m",
"id01973": "m",
"id01974": "m",
"id01975": "m",
"id01976": "m",
"id01979": "m",
"id01981": "m",
"id01982": "f",
"id01985": "m",
"id01986": "m",
"id01987": "m",
"id01988": "m",
"id01989": "m",
"id01990": "m",
"id01992": "m",
"id01993": "m",
"id01994": "m",
"id01995": "m",
"id01996": "m",
"id01997": "m",
"id01999": "m",
"id02000": "m",
"id02001": "m",
"id02004": "m",
"id02005": "m",
"id02006": "m",
"id02007": "m",
"id02008": "m",
"id02009": "m",
"id02010": "m",
"id02011": "m",
"id02013": "m",
"id02014": "m",
"id02015": "m",
"id02016": "m",
"id02017": "m",
"id02018": "m",
"id02019": "m",
"id02020": "m",
"id02021": "m",
"id02022": "m",
"id02023": "m",
"id02024": "m",
"id02030": "m",
"id02032": "m",
"id02033": "m",
"id02034": "m",
"id02035": "m",
"id02036": "m",
"id02037": "m",
"id02038": "m",
"id02039": "m",
"id02040": "m",
"id02041": "m",
"id02043": "m",
"id02045": "m",
"id02047": "m",
"id02048": "m",
"id02049": "m",
"id02051": "m",
"id02052": "m",
"id02053": "m",
"id02055": "m",
"id02057": "m",
"id02058": "m",
"id02059": "m",
"id02060": "m",
"id02061": "m",
"id02062": "m",
"id02067": "f",
"id02068": "m",
"id02070": "m",
"id02071": "f",
"id02072": "m",
"id02073": "m",
"id02074": "f",
"id02075": "m",
"id02077": "m",
"id02078": "m",
"id02079": "m",
"id02080": "m",
"id02081": "m",
"id02083": "m",
"id02084": "f",
"id02085": "f",
"id02086": "f",
"id02087": "f",
"id02088": "f",
"id02089": "f",
"id02090": "f",
"id02091": "f",
"id02092": "f",
"id02093": "f",
"id02094": "f",
"id02095": "f",
"id02096": "f",
"id02097": "m",
"id02098": "m",
"id02099": "f",
"id02100": "f",
"id02101": "f",
"id02102": "m",
"id02103": "f",
"id02104": "f",
"id02105": "f",
"id02106": "m",
"id02108": "m",
"id02109": "m",
"id02110": "f",
"id02111": "m",
"id02112": "f",
"id02113": "f",
"id02114": "f",
"id02115": "f",
"id02116": "f",
"id02119": "f",
"id02120": "m",
"id02121": "f",
"id02122": "m",
"id02123": "m",
"id02124": "m",
"id02133": "f",
"id02134": "f",
"id02139": "m",
"id02140": "m",
"id02141": "m",
"id02147": "m",
"id02148": "m",
"id02149": "m",
"id02150": "m",
"id02151": "m",
"id02152": "m",
"id02153": "f",
"id02155": "m",
"id02156": "m",
"id02157": "m",
"id02158": "f",
"id02159": "f",
"id02160": "m",
"id02162": "m",
"id02163": "m",
"id02164": "m",
"id02166": "f",
"id02168": "f",
"id02172": "f",
"id02173": "f",
"id02174": "f",
"id02175": "f",
"id02176": "f",
"id02177": "f",
"id02178": "f",
"id02179": "f",
"id02180": "f",
"id02181": "f",
"id02183": "m",
"id02185": "m",
"id02186": "m",
"id02187": "m",
"id02188": "m",
"id02189": "m",
"id02191": "m",
"id02193": "m",
"id02194": "m",
"id02196": "m",
"id02197": "m",
"id02201": "m",
"id02202": "m",
"id02203": "m",
"id02204": "m",
"id02205": "m",
"id02206": "m",
"id02208": "m",
"id02209": "m",
"id02210": "m",
"id02211": "m",
"id02212": "m",
"id02213": "m",
"id02214": "m",
"id02215": "m",
"id02216": "m",
"id02219": "f",
"id02220": "f",
"id02222": "f",
"id02223": "f",
"id02226": "m",
"id02228": "f",
"id02229": "m",
"id02230": "m",
"id02231": "f",
"id02233": "m",
"id02234": "m",
"id02235": "m",
"id02236": "m",
"id02239": "f",
"id02240": "m",
"id02242": "m",
"id02243": "m",
"id02245": "m",
"id02247": "f",
"id02249": "m",
"id02252": "m",
"id02253": "m",
"id02254": "m",
"id02255": "f",
"id02257": "m",
"id02258": "m",
"id02260": "m",
"id02262": "m",
"id02263": "m",
"id02264": "m",
"id02266": "m",
"id02267": "m",
"id02268": "m",
"id02270": "m",
"id02271": "f",
"id02273": "f",
"id02275": "f",
"id02276": "f",
"id02281": "f",
"id02282": "m",
"id02286": "f",
"id02287": "m",
"id02288": "m",
"id02289": "m",
"id02291": "f",
"id02292": "f",
"id02293": "m",
"id02294": "m",
"id02295": "m",
"id02296": "m",
"id02297": "m",
"id02298": "f",
"id02299": "f",
"id02300": "m",
"id02301": "f",
"id02302": "m",
"id02303": "m",
"id02304": "f",
"id02306": "m",
"id02310": "f",
"id02311": "m",
"id02312": "m",
"id02314": "m",
"id02315": "m",
"id02316": "m",
"id02317": "m",
"id02318": "m",
"id02322": "m",
"id02323": "m",
"id02325": "m",
"id02327": "f",
"id02329": "f",
"id02331": "f",
"id02332": "m",
"id02338": "m",
"id02339": "m",
"id02340": "m",
"id02341": "m",
"id02342": "m",
"id02343": "m",
"id02344": "m",
"id02346": "m",
"id02348": "m",
"id02349": "m",
"id02350": "m",
"id02351": "m",
"id02353": "m",
"id02355": "m",
"id02356": "m",
"id02357": "m",
"id02358": "m",
"id02359": "m",
"id02361": "f",
"id02362": "m",
"id02363": "m",
"id02364": "m",
"id02365": "m",
"id02367": "f",
"id02368": "f",
"id02381": "m",
"id02384": "m",
"id02388": "f",
"id02389": "m",
"id02390": "m",
"id02391": "m",
"id02392": "m",
"id02395": "f",
"id02409": "f",
"id02410": "m",
"id02411": "f",
"id02413": "m",
"id02414": "m",
"id02416": "m",
"id02418": "m",
"id02420": "f",
"id02421": "f",
"id02426": "f",
"id02432": "f",
"id02436": "m",
"id02437": "f",
"id02438": "m",
"id02439": "m",
"id02440": "m",
"id02441": "f",
"id02444": "f",
"id02445": "f",
"id02446": "f",
"id02448": "f",
"id02449": "f",
"id02450": "f",
"id02452": "f",
"id02453": "f",
"id02455": "f",
"id02457": "f",
"id02458": "f",
"id02459": "f",
"id02460": "f",
"id02461": "f",
"id02462": "f",
"id02463": "f",
"id02464": "f",
"id02465": "f",
"id02466": "f",
"id02467": "f",
"id02469": "m",
"id02471": "f",
"id02472": "f",
"id02473": "f",
"id02475": "f",
"id02477": "f",
"id02479": "f",
"id02482": "f",
"id02484": "f",
"id02485": "f",
"id02486": "m",
"id02488": "m",
"id02489": "f",
"id02490": "f",
"id02493": "m",
"id02494": "m",
"id02495": "m",
"id02496": "m",
"id02497": "m",
"id02499": "m",
"id02501": "m",
"id02502": "f",
"id02505": "m",
"id02508": "f",
"id02509": "m",
"id02511": "m",
"id02513": "m",
"id02515": "m",
"id02530": "f",
"id02531": "f",
"id02532": "f",
"id02534": "f",
"id02535": "f",
"id02537": "f",
"id02538": "f",
"id02539": "f",
"id02540": "f",
"id02542": "m",
"id02544": "m",
"id02545": "m",
"id02546": "f",
"id02547": "f",
"id02548": "f",
"id02553": "m",
"id02554": "m",
"id02555": "m",
"id02558": "m",
"id02561": "m",
"id02562": "m",
"id02563": "m",
"id02565": "m",
"id02566": "m",
"id02567": "m",
"id02568": "m",
"id02569": "m",
"id02570": "m",
"id02572": "m",
"id02573": "m",
"id02574": "m",
"id02575": "m",
"id02576": "m",
"id02577": "m",
"id02578": "m",
"id02579": "m",
"id02580": "m",
"id02583": "m",
"id02584": "m",
"id02585": "f",
"id02586": "f",
"id02587": "f",
"id02590": "m",
"id02591": "m",
"id02592": "m",
"id02593": "m",
"id02596": "f",
"id02597": "f",
"id02598": "f",
"id02599": "f",
"id02607": "m",
"id02608": "m",
"id02609": "f",
"id02613": "m",
"id02615": "m",
"id02617": "f",
"id02619": "f",
"id02622": "f",
"id02623": "f",
"id02625": "m",
"id02627": "f",
"id02628": "m",
"id02629": "m",
"id02631": "f",
"id02633": "f",
"id02634": "f",
"id02636": "m",
"id02638": "m",
"id02639": "f",
"id02640": "m",
"id02641": "f",
"id02642": "f",
"id02643": "m",
"id02647": "m",
"id02650": "f",
"id02653": "f",
"id02654": "f",
"id02655": "f",
"id02656": "f",
"id02658": "f",
"id02659": "f",
"id02660": "m",
"id02661": "m",
"id02662": "m",
"id02665": "m",
"id02666": "f",
"id02667": "f",
"id02668": "f",
"id02669": "f",
"id02672": "f",
"id02673": "f",
"id02680": "f",
"id02683": "m",
"id02684": "m",
"id02685": "m",
"id02687": "m",
"id02693": "m",
"id02695": "m",
"id02696": "m",
"id02698": "f",
"id02700": "m",
"id02702": "m",
"id02703": "m",
"id02705": "m",
"id02706": "m",
"id02707": "m",
"id02710": "m",
"id02711": "m",
"id02713": "m",
"id02714": "m",
"id02715": "m",
"id02719": "f",
"id02720": "f",
"id02721": "f",
"id02722": "f",
"id02724": "f",
"id02725": "f",
"id02728": "f",
"id02729": "f",
"id02730": "f",
"id02731": "m",
"id02733": "m",
"id02734": "m",
"id02735": "m",
"id02736": "m",
"id02738": "m",
"id02739": "m",
"id02740": "m",
"id02745": "m",
"id02746": "f",
"id02747": "f",
"id02751": "m",
"id02759": "f",
"id02760": "f",
"id02761": "f",
"id02764": "m",
"id02765": "m",
"id02766": "m",
"id02780": "f",
"id02781": "f",
"id02783": "m",
"id02784": "m",
"id02786": "m",
"id02787": "m",
"id02790": "m",
"id02791": "m",
"id02793": "m",
"id02795": "f",
"id02805": "f",
"id02806": "f",
"id02807": "f",
"id02808": "f",
"id02809": "f",
"id02810": "m",
"id02812": "m",
"id02815": "m",
"id02816": "m",
"id02817": "f",
"id02818": "m",
"id02820": "m",
"id02821": "f",
"id02823": "f",
"id02824": "f",
"id02825": "f",
"id02827": "m",
"id02829": "m",
"id02830": "m",
"id02832": "m",
"id02833": "m",
"id02836": "m",
"id02837": "m",
"id02838": "f",
"id02839": "f",
"id02840": "m",
"id02844": "f",
"id02845": "f",
"id02846": "f",
"id02847": "f",
"id02848": "f",
"id02851": "m",
"id02853": "m",
"id02854": "m",
"id02857": "m",
"id02858": "m",
"id02863": "m",
"id02864": "m",
"id02865": "m",
"id02866": "m",
"id02869": "m",
"id02871": "m",
"id02872": "m",
"id02873": "m",
"id02874": "m",
"id02875": "m",
"id02876": "m",
"id02877": "m",
"id02878": "m",
"id02879": "m",
"id02880": "m",
"id02881": "m",
"id02882": "f",
"id02883": "m",
"id02884": "m",
"id02886": "m",
"id02887": "m",
"id02888": "m",
"id02889": "m",
"id02890": "m",
"id02891": "m",
"id02892": "m",
"id02893": "m",
"id02896": "m",
"id02897": "m",
"id02898": "m",
"id02899": "m",
"id02900": "m",
"id02901": "m",
"id02902": "m",
"id02903": "m",
"id02904": "m",
"id02905": "m",
"id02908": "m",
"id02909": "m",
"id02911": "m",
"id02912": "f",
"id02914": "f",
"id02916": "m",
"id02917": "m",
"id02918": "m",
"id02920": "m",
"id02924": "f",
"id02926": "m",
"id02928": "f",
"id02935": "m",
"id02936": "m",
"id02939": "m",
"id02940": "m",
"id02942": "f",
"id02943": "m",
"id02944": "m",
"id02945": "f",
"id02946": "f",
"id02948": "f",
"id02950": "f",
"id02952": "m",
"id02953": "f",
"id02954": "f",
"id02955": "f",
"id02956": "f",
"id02957": "m",
"id02958": "f",
"id02959": "m",
"id02961": "m",
"id02963": "m",
"id02964": "m",
"id02965": "m",
"id02978": "m",
"id02979": "m",
"id02980": "m",
"id02981": "m",
"id02982": "m",
"id02983": "m",
"id02984": "m",
"id02986": "m",
"id02987": "m",
"id02988": "m",
"id02989": "m",
"id02991": "f",
"id02993": "m",
"id02995": "m",
"id02996": "f",
"id02998": "f",
"id02999": "m",
"id03000": "f",
"id03001": "f",
"id03002": "f",
"id03003": "f",
"id03004": "f",
"id03005": "m",
"id03006": "f",
"id03007": "f",
"id03009": "m",
"id03013": "m",
"id03014": "m",
"id03015": "m",
"id03016": "m",
"id03017": "m",
"id03018": "m",
"id03020": "m",
"id03026": "m",
"id03027": "m",
"id03028": "m",
"id03029": "m",
"id03030": "m",
"id03031": "f",
"id03037": "m",
"id03038": "m",
"id03039": "m",
"id03040": "m",
"id03041": "m",
"id03042": "m",
"id03043": "m",
"id03056": "m",
"id03057": "m",
"id03059": "f",
"id03060": "f",
"id03064": "f",
"id03066": "m",
"id03067": "m",
"id03070": "m",
"id03071": "m",
"id03073": "m",
"id03074": "m",
"id03075": "m",
"id03077": "m",
"id03078": "m",
"id03079": "m",
"id03082": "f",
"id03084": "m",
"id03087": "m",
"id03088": "m",
"id03089": "m",
"id03090": "m",
"id03091": "f",
"id03092": "m",
"id03098": "f",
"id03099": "f",
"id03100": "f",
"id03101": "f",
"id03102": "f",
"id03103": "f",
"id03107": "m",
"id03109": "m",
"id03110": "f",
"id03111": "f",
"id03112": "m",
"id03115": "m",
"id03118": "m",
"id03119": "m",
"id03123": "f",
"id03124": "f",
"id03125": "m",
"id03127": "f",
"id03129": "f",
"id03133": "f",
"id03134": "m",
"id03136": "m",
"id03137": "m",
"id03145": "m",
"id03146": "m",
"id03147": "f",
"id03148": "f",
"id03149": "f",
"id03150": "m",
"id03158": "f",
"id03168": "m",
"id03174": "m",
"id03175": "f",
"id03176": "m",
"id03177": "m",
"id03178": "m",
"id03179": "m",
"id03180": "m",
"id03182": "f",
"id03183": "m",
"id03184": "f",
"id03187": "m",
"id03188": "m",
"id03189": "m",
"id03190": "m",
"id03192": "m",
"id03193": "m",
"id03194": "m",
"id03195": "m",
"id03196": "m",
"id03198": "m",
"id03199": "m",
"id03200": "m",
"id03201": "m",
"id03202": "m",
"id03203": "m",
"id03205": "m",
"id03207": "m",
"id03209": "m",
"id03210": "m",
"id03211": "f",
"id03212": "f",
"id03213": "f",
"id03214": "f",
"id03215": "m",
"id03216": "m",
"id03220": "f",
"id03221": "m",
"id03223": "m",
"id03226": "m",
"id03227": "m",
"id03228": "m",
"id03229": "m",
"id03230": "m",
"id03232": "m",
"id03236": "m",
"id03238": "m",
"id03240": "m",
"id03241": "m",
"id03242": "m",
"id03243": "m",
"id03244": "m",
"id03245": "m",
"id03246": "m",
"id03247": "m",
"id03248": "m",
"id03249": "m",
"id03250": "m",
"id03253": "m",
"id03258": "m",
"id03260": "m",
"id03261": "m",
"id03262": "m",
"id03263": "m",
"id03264": "m",
"id03265": "m",
"id03266": "m",
"id03267": "m",
"id03269": "m",
"id03272": "f",
"id03273": "f",
"id03274": "m",
"id03279": "m",
"id03281": "f",
"id03283": "m",
"id03284": "m",
"id03285": "m",
"id03286": "m",
"id03287": "m",
"id03288": "m",
"id03289": "f",
"id03290": "f",
"id03291": "f",
"id03292": "m",
"id03293": "m",
"id03295": "m",
"id03298": "m",
"id03299": "m",
"id03300": "f",
"id03303": "f",
"id03304": "f",
"id03306": "m",
"id03307": "f",
"id03312": "m",
"id03313": "m",
"id03314": "m",
"id03315": "f",
"id03316": "f",
"id03317": "f",
"id03318": "f",
"id03319": "f",
"id03320": "f",
"id03323": "f",
"id03324": "f",
"id03325": "f",
"id03326": "m",
"id03327": "m",
"id03328": "m",
"id03329": "m",
"id03331": "m",
"id03332": "m",
"id03333": "m",
"id03334": "m",
"id03335": "m",
"id03336": "f",
"id03337": "m",
"id03338": "m",
"id03339": "f",
"id03340": "m",
"id03341": "m",
"id03342": "m",
"id03343": "m",
"id03344": "m",
"id03345": "f",
"id03346": "m",
"id03347": "m",
"id03348": "m",
"id03349": "m",
"id03350": "m",
"id03351": "m",
"id03352": "m",
"id03353": "m",
"id03354": "m",
"id03360": "m",
"id03361": "m",
"id03363": "m",
"id03364": "m",
"id03366": "m",
"id03371": "f",
"id03372": "f",
"id03373": "f",
"id03374": "f",
"id03375": "f",
"id03377": "m",
"id03379": "f",
"id03381": "f",
"id03382": "f",
"id03383": "f",
"id03384": "f",
"id03386": "f",
"id03387": "f",
"id03388": "f",
"id03389": "f",
"id03390": "f",
"id03398": "m",
"id03399": "m",
"id03400": "f",
"id03401": "f",
"id03402": "f",
"id03405": "f",
"id03409": "m",
"id03410": "f",
"id03412": "m",
"id03413": "m",
"id03414": "m",
"id03417": "m",
"id03422": "m",
"id03424": "m",
"id03425": "m",
"id03426": "m",
"id03427": "m",
"id03428": "m",
"id03429": "m",
"id03430": "m",
"id03432": "m",
"id03433": "m",
"id03434": "m",
"id03435": "m",
"id03436": "m",
"id03439": "m",
"id03443": "m",
"id03444": "f",
"id03445": "m",
"id03446": "f",
"id03448": "m",
"id03449": "m",
"id03452": "f",
"id03453": "f",
"id03454": "f",
"id03455": "f",
"id03457": "f",
"id03458": "f",
"id03459": "f",
"id03460": "f",
"id03461": "f",
"id03462": "m",
"id03463": "m",
"id03467": "m",
"id03471": "f",
"id03477": "f",
"id03479": "f",
"id03480": "m",
"id03483": "m",
"id03485": "m",
"id03486": "m",
"id03487": "m",
"id03488": "m",
"id03489": "m",
"id03493": "f",
"id03495": "m",
"id03497": "m",
"id03499": "m",
"id03501": "m",
"id03502": "f",
"id03503": "m",
"id03506": "m",
"id03508": "m",
"id03509": "m",
"id03511": "m",
"id03512": "m",
"id03514": "m",
"id03515": "m",
"id03517": "f",
"id03518": "f",
"id03519": "m",
"id03520": "m",
"id03521": "m",
"id03522": "m",
"id03523": "m",
"id03524": "m",
"id03525": "m",
"id03526": "m",
"id03527": "m",
"id03528": "m",
"id03529": "m",
"id03530": "m",
"id03531": "m",
"id03532": "m",
"id03534": "m",
"id03535": "f",
"id03537": "m",
"id03538": "m",
"id03540": "m",
"id03543": "m",
"id03546": "m",
"id03548": "m",
"id03550": "m",
"id03556": "f",
"id03557": "m",
"id03559": "f",
"id03560": "m",
"id03569": "f",
"id03570": "f",
"id03572": "f",
"id03573": "m",
"id03575": "f",
"id03577": "f",
"id03578": "m",
"id03580": "f",
"id03581": "f",
"id03582": "f",
"id03584": "f",
"id03587": "f",
"id03588": "m",
"id03589": "f",
"id03592": "f",
"id03597": "f",
"id03599": "m",
"id03605": "f",
"id03606": "f",
"id03609": "m",
"id03613": "m",
"id03614": "m",
"id03615": "f",
"id03617": "f",
"id03618": "f",
"id03619": "f",
"id03620": "f",
"id03621": "f",
"id03622": "f",
"id03623": "m",
"id03624": "m",
"id03625": "f",
"id03628": "m",
"id03631": "m",
"id03632": "m",
"id03633": "m",
"id03634": "m",
"id03639": "f",
"id03640": "m",
"id03643": "m",
"id03644": "m",
"id03645": "m",
"id03647": "m",
"id03649": "f",
"id03650": "f",
"id03651": "f",
"id03652": "m",
"id03654": "m",
"id03656": "f",
"id03658": "m",
"id03659": "f",
"id03661": "m",
"id03663": "m",
"id03664": "m",
"id03665": "m",
"id03667": "m",
"id03668": "m",
"id03669": "m",
"id03670": "m",
"id03671": "m",
"id03672": "m",
"id03673": "m",
"id03675": "m",
"id03677": "m",
"id03678": "m",
"id03679": "m",
"id03680": "m",
"id03681": "m",
"id03682": "m",
"id03683": "m",
"id03684": "m",
"id03685": "m",
"id03686": "m",
"id03687": "m",
"id03688": "m",
"id03689": "m",
"id03690": "m",
"id03692": "m",
"id03693": "m",
"id03694": "m",
"id03696": "f",
"id03699": "m",
"id03701": "m",
"id03702": "f",
"id03703": "m",
"id03704": "m",
"id03705": "m",
"id03706": "m",
"id03707": "f",
"id03709": "m",
"id03710": "m",
"id03711": "m",
"id03713": "f",
"id03714": "m",
"id03715": "f",
"id03716": "f",
"id03717": "f",
"id03719": "m",
"id03720": "m",
"id03721": "m",
"id03722": "f",
"id03723": "f",
"id03724": "m",
"id03725": "f",
"id03727": "f",
"id03728": "m",
"id03729": "m",
"id03730": "m",
"id03731": "m",
"id03732": "m",
"id03733": "m",
"id03735": "m",
"id03736": "m",
"id03740": "m",
"id03741": "m",
"id03743": "m",
"id03744": "m",
"id03745": "m",
"id03746": "m",
"id03747": "f",
"id03748": "m",
"id03749": "m",
"id03750": "m",
"id03751": "m",
"id03752": "m",
"id03753": "m",
"id03754": "m",
"id03755": "m",
"id03756": "m",
"id03757": "m",
"id03758": "m",
"id03759": "m",
"id03760": "m",
"id03761": "m",
"id03762": "m",
"id03763": "m",
"id03764": "m",
"id03765": "m",
"id03767": "m",
"id03768": "m",
"id03769": "m",
"id03770": "m",
"id03772": "m",
"id03773": "m",
"id03774": "m",
"id03775": "m",
"id03777": "m",
"id03778": "m",
"id03779": "f",
"id03780": "m",
"id03781": "f",
"id03782": "m",
"id03784": "m",
"id03785": "m",
"id03786": "f",
"id03788": "f",
"id03789": "m",
"id03790": "m",
"id03791": "m",
"id03795": "m",
"id03796": "m",
"id03797": "m",
"id03800": "m",
"id03803": "m",
"id03804": "m",
"id03810": "m",
"id03811": "m",
"id03812": "f",
"id03813": "f",
"id03815": "f",
"id03816": "f",
"id03817": "f",
"id03818": "f",
"id03819": "f",
"id03820": "f",
"id03821": "f",
"id03822": "f",
"id03823": "m",
"id03824": "m",
"id03825": "f",
"id03827": "f",
"id03828": "f",
"id03829": "f",
"id03830": "m",
"id03837": "m",
"id03838": "m",
"id03839": "m",
"id03840": "m",
"id03841": "m",
"id03842": "m",
"id03843": "m",
"id03844": "f",
"id03845": "m",
"id03846": "m",
"id03847": "m",
"id03848": "m",
"id03852": "m",
"id03853": "m",
"id03854": "m",
"id03855": "m",
"id03856": "f",
"id03858": "f",
"id03859": "f",
"id03860": "m",
"id03861": "m",
"id03862": "m",
"id03863": "m",
"id03864": "m",
"id03865": "m",
"id03866": "m",
"id03867": "m",
"id03868": "m",
"id03869": "m",
"id03870": "m",
"id03871": "m",
"id03873": "m",
"id03876": "m",
"id03877": "m",
"id03878": "m",
"id03884": "m",
"id03885": "m",
"id03887": "m",
"id03888": "m",
"id03889": "m",
"id03890": "m",
"id03891": "m",
"id03892": "m",
"id03893": "m",
"id03894": "m",
"id03895": "m",
"id03896": "f",
"id03897": "f",
"id03898": "m",
"id03899": "m",
"id03900": "m",
"id03902": "m",
"id03903": "f",
"id03904": "m",
"id03905": "m",
"id03906": "m",
"id03907": "m",
"id03909": "m",
"id03910": "m",
"id03912": "m",
"id03913": "m",
"id03914": "m",
"id03915": "m",
"id03916": "m",
"id03917": "m",
"id03918": "m",
"id03919": "m",
"id03920": "m",
"id03921": "m",
"id03922": "m",
"id03923": "m",
"id03924": "m",
"id03927": "m",
"id03928": "m",
"id03929": "m",
"id03930": "m",
"id03931": "m",
"id03932": "m",
"id03934": "m",
"id03935": "m",
"id03936": "m",
"id03938": "m",
"id03939": "m",
"id03940": "f",
"id03941": "f",
"id03942": "f",
"id03943": "f",
"id03945": "m",
"id03946": "m",
"id03947": "m",
"id03949": "m",
"id03950": "m",
"id03951": "m",
"id03952": "m",
"id03953": "m",
"id03954": "m",
"id03955": "m",
"id03956": "m",
"id03958": "m",
"id03961": "f",
"id03962": "f",
"id03963": "f",
"id03965": "m",
"id03966": "m",
"id03967": "f",
"id03968": "f",
"id03969": "f",
"id03970": "f",
"id03971": "f",
"id03972": "f",
"id03974": "f",
"id03976": "f",
"id03977": "f",
"id03978": "f",
"id03980": "f",
"id03981": "f",
"id03983": "f",
"id03984": "f",
"id03985": "f",
"id03986": "f",
"id03987": "f",
"id03989": "f",
"id03991": "f",
"id03992": "m",
"id03993": "m",
"id03994": "m",
"id03996": "m",
"id03998": "m",
"id03999": "m",
"id04005": "m",
"id04006": "m",
"id04007": "m",
"id04009": "m",
"id04010": "m",
"id04012": "m",
"id04013": "m",
"id04014": "m",
"id04015": "m",
"id04016": "m",
"id04017": "m",
"id04018": "m",
"id04019": "m",
"id04021": "m",
"id04022": "m",
"id04024": "m",
"id04025": "m",
"id04026": "f",
"id04027": "f",
"id04028": "f",
"id04029": "m",
"id04030": "f",
"id04031": "m",
"id04032": "m",
"id04033": "m",
"id04034": "m",
"id04035": "f",
"id04036": "f",
"id04037": "f",
"id04038": "f",
"id04039": "f",
"id04042": "f",
"id04044": "f",
"id04045": "f",
"id04046": "f",
"id04047": "f",
"id04048": "f",
"id04049": "f",
"id04050": "f",
"id04051": "f",
"id04052": "f",
"id04053": "f",
"id04054": "f",
"id04055": "f",
"id04057": "f",
"id04058": "m",
"id04062": "m",
"id04063": "m",
"id04064": "f",
"id04065": "f",
"id04066": "f",
"id04070": "f",
"id04073": "f",
"id04074": "f",
"id04076": "f",
"id04077": "f",
"id04079": "m",
"id04080": "m",
"id04081": "m",
"id04082": "m",
"id04084": "m",
"id04085": "m",
"id04086": "m",
"id04089": "m",
"id04091": "m",
"id04092": "m",
"id04093": "m",
"id04094": "m",
"id04095": "m",
"id04096": "m",
"id04097": "m",
"id04098": "m",
"id04100": "m",
"id04102": "m",
"id04103": "m",
"id04105": "m",
"id04106": "m",
"id04107": "m",
"id04108": "m",
"id04109": "m",
"id04111": "m",
"id04114": "m",
"id04115": "m",
"id04116": "m",
"id04117": "m",
"id04118": "m",
"id04119": "m",
"id04121": "m",
"id04122": "f",
"id04125": "m",
"id04126": "m",
"id04127": "m",
"id04128": "m",
"id04129": "m",
"id04130": "m",
"id04132": "f",
"id04133": "m",
"id04134": "f",
"id04136": "f",
"id04137": "m",
"id04140": "f",
"id04144": "f",
"id04145": "m",
"id04146": "m",
"id04147": "m",
"id04148": "m",
"id04149": "f",
"id04150": "f",
"id04151": "f",
"id04152": "f",
"id04153": "f",
"id04154": "m",
"id04155": "m",
"id04157": "m",
"id04159": "m",
"id04160": "m",
"id04161": "m",
"id04162": "m",
"id04163": "m",
"id04164": "m",
"id04165": "m",
"id04166": "m",
"id04167": "m",
"id04168": "m",
"id04169": "m",
"id04170": "m",
"id04171": "m",
"id04173": "m",
"id04174": "f",
"id04175": "f",
"id04177": "m",
"id04178": "f",
"id04179": "m",
"id04180": "m",
"id04181": "m",
"id04182": "m",
"id04183": "m",
"id04184": "f",
"id04186": "f",
"id04187": "f",
"id04188": "f",
"id04189": "f",
"id04190": "f",
"id04191": "m",
"id04192": "m",
"id04193": "m",
"id04194": "m",
"id04195": "m",
"id04196": "m",
"id04197": "m",
"id04198": "m",
"id04199": "m",
"id04202": "m",
"id04203": "m",
"id04204": "m",
"id04205": "m",
"id04206": "m",
"id04207": "m",
"id04215": "m",
"id04216": "m",
"id04219": "m",
"id04221": "m",
"id04222": "m",
"id04223": "m",
"id04224": "m",
"id04225": "m",
"id04227": "m",
"id04228": "m",
"id04229": "m",
"id04230": "m",
"id04231": "m",
"id04232": "m",
"id04234": "m",
"id04235": "m",
"id04236": "m",
"id04237": "m",
"id04238": "m",
"id04239": "m",
"id04240": "m",
"id04243": "m",
"id04244": "m",
"id04245": "f",
"id04246": "m",
"id04247": "f",
"id04249": "f",
"id04250": "m",
"id04251": "m",
"id04253": "m",
"id04254": "m",
"id04255": "m",
"id04256": "m",
"id04257": "m",
"id04258": "m",
"id04259": "m",
"id04260": "m",
"id04261": "m",
"id04262": "m",
"id04263": "m",
"id04264": "m",
"id04265": "m",
"id04266": "m",
"id04267": "m",
"id04268": "m",
"id04269": "m",
"id04270": "m",
"id04271": "m",
"id04272": "m",
"id04273": "m",
"id04274": "m",
"id04275": "m",
"id04276": "m",
"id04278": "m",
"id04279": "m",
"id04280": "m",
"id04282": "m",
"id04284": "f",
"id04285": "m",
"id04287": "m",
"id04288": "m",
"id04289": "m",
"id04292": "m",
"id04293": "m",
"id04294": "m",
"id04295": "m",
"id04296": "m",
"id04297": "m",
"id04298": "m",
"id04300": "m",
"id04301": "m",
"id04302": "m",
"id04303": "m",
"id04304": "m",
"id04305": "m",
"id04309": "m",
"id04311": "m",
"id04312": "m",
"id04313": "m",
"id04314": "m",
"id04316": "m",
"id04318": "f",
"id04319": "m",
"id04320": "m",
"id04321": "m",
"id04322": "f",
"id04325": "m",
"id04326": "m",
"id04332": "m",
"id04333": "m",
"id04334": "m",
"id04335": "m",
"id04336": "m",
"id04340": "m",
"id04341": "m",
"id04342": "m",
"id04343": "m",
"id04344": "f",
"id04345": "m",
"id04346": "m",
"id04349": "m",
"id04350": "m",
"id04352": "m",
"id04353": "m",
"id04355": "m",
"id04356": "m",
"id04357": "m",
"id04358": "m",
"id04365": "m",
"id04366": "m",
"id04367": "m",
"id04368": "m",
"id04370": "m",
"id04372": "m",
"id04373": "f",
"id04374": "f",
"id04375": "f",
"id04376": "f",
"id04377": "f",
"id04378": "f",
"id04379": "f",
"id04380": "f",
"id04383": "m",
"id04386": "m",
"id04387": "m",
"id04388": "m",
"id04392": "m",
"id04393": "m",
"id04396": "m",
"id04397": "m",
"id04398": "m",
"id04399": "m",
"id04402": "m",
"id04404": "m",
"id04405": "f",
"id04406": "m",
"id04407": "m",
"id04408": "m",
"id04409": "f",
"id04413": "f",
"id04414": "f",
"id04415": "f",
"id04416": "f",
"id04417": "f",
"id04418": "m",
"id04419": "f",
"id04420": "f",
"id04421": "m",
"id04422": "f",
"id04423": "f",
"id04424": "f",
"id04425": "f",
"id04427": "f",
"id04428": "f",
"id04429": "f",
"id04430": "f",
"id04432": "f",
"id04433": "f",
"id04434": "m",
"id04436": "m",
"id04437": "f",
"id04441": "f",
"id04442": "m",
"id04443": "f",
"id04444": "f",
"id04445": "f",
"id04446": "f",
"id04447": "f",
"id04449": "f",
"id04450": "f",
"id04451": "f",
"id04452": "f",
"id04453": "m",
"id04455": "m",
"id04456": "f",
"id04458": "f",
"id04459": "m",
"id04460": "m",
"id04462": "m",
"id04463": "m",
"id04474": "f",
"id04475": "m",
"id04476": "m",
"id04477": "m",
"id04478": "m",
"id04479": "m",
"id04481": "m",
"id04482": "m",
"id04483": "m",
"id04484": "m",
"id04485": "m",
"id04486": "m",
"id04487": "m",
"id04488": "f",
"id04489": "f",
"id04490": "f",
"id04491": "f",
"id04493": "m",
"id04494": "m",
"id04495": "m",
"id04496": "m",
"id04497": "m",
"id04498": "m",
"id04499": "m",
"id04501": "m",
"id04502": "m",
"id04503": "m",
"id04504": "m",
"id04505": "m",
"id04506": "m",
"id04508": "m",
"id04509": "m",
"id04510": "m",
"id04511": "m",
"id04512": "m",
"id04513": "m",
"id04514": "m",
"id04516": "f",
"id04517": "m",
"id04518": "f",
"id04519": "m",
"id04520": "f",
"id04522": "f",
"id04523": "m",
"id04524": "m",
"id04525": "f",
"id04526": "m",
"id04527": "f",
"id04528": "f",
"id04529": "f",
"id04530": "f",
"id04531": "m",
"id04534": "f",
"id04536": "m",
"id04537": "m",
"id04538": "f",
"id04540": "f",
"id04542": "m",
"id04543": "m",
"id04545": "f",
"id04547": "f",
"id04549": "m",
"id04551": "m",
"id04552": "m",
"id04554": "m",
"id04555": "f",
"id04556": "f",
"id04557": "f",
"id04559": "f",
"id04560": "m",
"id04561": "m",
"id04562": "m",
"id04563": "m",
"id04564": "f",
"id04565": "m",
"id04566": "m",
"id04567": "f",
"id04569": "f",
"id04570": "f",
"id04571": "f",
"id04573": "f",
"id04576": "f",
"id04578": "m",
"id04579": "f",
"id04580": "f",
"id04582": "f",
"id04583": "f",
"id04584": "f",
"id04585": "m",
"id04586": "m",
"id04588": "m",
"id04589": "m",
"id04590": "f",
"id04593": "m",
"id04597": "f",
"id04599": "m",
"id04600": "f",
"id04601": "m",
"id04602": "m",
"id04606": "m",
"id04608": "m",
"id04609": "f",
"id04610": "f",
"id04613": "f",
"id04614": "f",
"id04615": "f",
"id04616": "f",
"id04617": "f",
"id04618": "f",
"id04619": "f",
"id04621": "f",
"id04623": "f",
"id04624": "f",
"id04625": "f",
"id04626": "f",
"id04627": "f",
"id04628": "f",
"id04629": "f",
"id04635": "f",
"id04637": "f",
"id04638": "f",
"id04639": "f",
"id04640": "f",
"id04641": "f",
"id04642": "f",
"id04643": "f",
"id04644": "f",
"id04645": "f",
"id04646": "f",
"id04647": "f",
"id04648": "f",
"id04649": "f",
"id04650": "f",
"id04651": "f",
"id04652": "f",
"id04653": "f",
"id04654": "f",
"id04655": "f",
"id04656": "f",
"id04657": "f",
"id04658": "f",
"id04660": "f",
"id04662": "f",
"id04665": "m",
"id04666": "f",
"id04667": "f",
"id04668": "f",
"id04675": "f",
"id04676": "f",
"id04677": "m",
"id04684": "m",
"id04686": "m",
"id04687": "m",
"id04689": "f",
"id04690": "f",
"id04691": "m",
"id04692": "m",
"id04693": "m",
"id04694": "f",
"id04695": "f",
"id04696": "f",
"id04697": "f",
"id04698": "f",
"id04699": "m",
"id04701": "f",
"id04703": "m",
"id04704": "f",
"id04705": "f",
"id04706": "f",
"id04708": "f",
"id04709": "m",
"id04710": "m",
"id04712": "m",
"id04713": "m",
"id04714": "m",
"id04715": "m",
"id04716": "m",
"id04717": "m",
"id04718": "m",
"id04720": "m",
"id04721": "m",
"id04723": "m",
"id04725": "m",
"id04726": "m",
"id04727": "m",
"id04728": "m",
"id04730": "f",
"id04731": "f",
"id04733": "f",
"id04734": "m",
"id04736": "f",
"id04737": "m",
"id04738": "m",
"id04739": "m",
"id04740": "m",
"id04741": "m",
"id04742": "m",
"id04743": "m",
"id04744": "m",
"id04745": "m",
"id04746": "m",
"id04747": "m",
"id04748": "m",
"id04750": "m",
"id04751": "m",
"id04752": "m",
"id04753": "m",
"id04754": "m",
"id04755": "m",
"id04756": "m",
"id04757": "m",
"id04758": "m",
"id04759": "m",
"id04761": "m",
"id04762": "m",
"id04763": "f",
"id04770": "m",
"id04774": "m",
"id04776": "m",
"id04778": "f",
"id04780": "m",
"id04781": "f",
"id04782": "f",
"id04783": "f",
"id04784": "m",
"id04787": "f",
"id04788": "f",
"id04789": "m",
"id04800": "f",
"id04801": "m",
"id04802": "f",
"id04803": "f",
"id04814": "f",
"id04818": "f",
"id04820": "f",
"id04821": "f",
"id04822": "f",
"id04823": "f",
"id04825": "m",
"id04826": "m",
"id04827": "f",
"id04830": "m",
"id04834": "m",
"id04835": "m",
"id04836": "m",
"id04837": "m",
"id04838": "f",
"id04839": "f",
"id04841": "m",
"id04842": "m",
"id04844": "f",
"id04845": "f",
"id04846": "f",
"id04847": "f",
"id04848": "f",
"id04851": "m",
"id04852": "m",
"id04854": "m",
"id04855": "m",
"id04856": "m",
"id04857": "m",
"id04858": "m",
"id04862": "m",
"id04863": "f",
"id04866": "m",
"id04869": "f",
"id04872": "f",
"id04873": "m",
"id04875": "m",
"id04876": "f",
"id04877": "m",
"id04884": "m",
"id04885": "m",
"id04886": "m",
"id04887": "m",
"id04888": "m",
"id04889": "m",
"id04890": "f",
"id04893": "f",
"id04894": "f",
"id04895": "f",
"id04896": "f",
"id04897": "f",
"id04898": "m",
"id04899": "m",
"id04900": "f",
"id04901": "f",
"id04902": "f",
"id04903": "f",
"id04904": "f",
"id04905": "f",
"id04906": "f",
"id04907": "f",
"id04908": "f",
"id04909": "f",
"id04910": "f",
"id04911": "m",
"id04914": "f",
"id04915": "f",
"id04917": "m",
"id04918": "m",
"id04926": "m",
"id04927": "f",
"id04928": "m",
"id04930": "m",
"id04933": "m",
"id04934": "m",
"id04935": "m",
"id04938": "m",
"id04939": "f",
"id04940": "m",
"id04941": "m",
"id04943": "m",
"id04945": "m",
"id04946": "m",
"id04948": "m",
"id04950": "f",
"id04951": "m",
"id04953": "m",
"id04956": "m",
"id04957": "m",
"id04958": "m",
"id04959": "f",
"id04961": "f",
"id04962": "f",
"id04963": "m",
"id04964": "f",
"id04967": "m",
"id04968": "f",
"id04969": "m",
"id04970": "f",
"id04971": "f",
"id04972": "f",
"id04973": "f",
"id04974": "m",
"id04976": "m",
"id04977": "m",
"id04978": "m",
"id04979": "f",
"id04980": "f",
"id04981": "m",
"id04982": "m",
"id04983": "m",
"id04984": "m",
"id04985": "m",
"id04986": "m",
"id04987": "f",
"id04988": "f",
"id04990": "f",
"id04991": "m",
"id04993": "f",
"id04994": "m",
"id04995": "m",
"id04996": "m",
"id04997": "m",
"id04998": "m",
"id04999": "m",
"id05000": "m",
"id05003": "f",
"id05004": "f",
"id05008": "f",
"id05009": "f",
"id05010": "f",
"id05011": "f",
"id05012": "f",
"id05013": "f",
"id05014": "f",
"id05015": "f",
"id05016": "f",
"id05018": "f",
"id05019": "f",
"id05020": "f",
"id05021": "f",
"id05022": "f",
"id05023": "f",
"id05025": "f",
"id05027": "m",
"id05028": "m",
"id05029": "f",
"id05030": "m",
"id05031": "m",
"id05032": "m",
"id05034": "m",
"id05035": "m",
"id05038": "m",
"id05039": "m",
"id05040": "f",
"id05041": "f",
"id05042": "f",
"id05044": "f",
"id05048": "m",
"id05049": "m",
"id05052": "m",
"id05054": "f",
"id05055": "m",
"id05056": "m",
"id05057": "f",
"id05058": "m",
"id05060": "f",
"id05061": "f",
"id05062": "m",
"id05063": "m",
"id05066": "m",
"id05067": "m",
"id05075": "m",
"id05076": "m",
"id05079": "m",
"id05082": "m",
"id05087": "f",
"id05088": "f",
"id05090": "m",
"id05091": "f",
"id05092": "m",
"id05093": "f",
"id05094": "m",
"id05095": "m",
"id05096": "f",
"id05099": "f",
"id05100": "m",
"id05101": "m",
"id05102": "m",
"id05103": "f",
"id05104": "f",
"id05105": "f",
"id05106": "f",
"id05107": "f",
"id05108": "f",
"id05121": "m",
"id05123": "m",
"id05124": "m",
"id05125": "m",
"id05129": "f",
"id05130": "m",
"id05131": "m",
"id05132": "m",
"id05133": "m",
"id05134": "f",
"id05135": "f",
"id05136": "m",
"id05137": "m",
"id05139": "m",
"id05141": "f",
"id05145": "m",
"id05147": "m",
"id05148": "m",
"id05149": "m",
"id05150": "m",
"id05152": "f",
"id05153": "f",
"id05154": "f",
"id05159": "m",
"id05160": "m",
"id05161": "f",
"id05162": "f",
"id05165": "m",
"id05166": "f",
"id05167": "m",
"id05168": "f",
"id05169": "f",
"id05170": "f",
"id05171": "f",
"id05172": "m",
"id05173": "f",
"id05174": "f",
"id05176": "f",
"id05178": "m",
"id05179": "m",
"id05181": "m",
"id05182": "f",
"id05186": "f",
"id05187": "f",
"id05188": "f",
"id05189": "m",
"id05190": "f",
"id05191": "f",
"id05194": "m",
"id05196": "m",
"id05197": "m",
"id05198": "m",
"id05199": "f",
"id05200": "f",
"id05201": "f",
"id05202": "f",
"id05203": "f",
"id05204": "f",
"id05205": "f",
"id05206": "f",
"id05207": "f",
"id05208": "f",
"id05209": "f",
"id05210": "f",
"id05211": "f",
"id05212": "f",
"id05213": "f",
"id05218": "f",
"id05220": "f",
"id05221": "f",
"id05223": "f",
"id05224": "f",
"id05226": "f",
"id05227": "f",
"id05228": "m",
"id05229": "m",
"id05231": "f",
"id05235": "f",
"id05236": "f",
"id05237": "f",
"id05239": "f",
"id05240": "f",
"id05241": "f",
"id05242": "m",
"id05244": "m",
"id05245": "f",
"id05246": "m",
"id05248": "f",
"id05249": "m",
"id05250": "f",
"id05251": "f",
"id05252": "f",
"id05253": "f",
"id05255": "f",
"id05256": "m",
"id05257": "m",
"id05258": "m",
"id05260": "m",
"id05261": "m",
"id05262": "m",
"id05263": "m",
"id05264": "m",
"id05265": "m",
"id05266": "m",
"id05268": "m",
"id05269": "m",
"id05270": "m",
"id05271": "m",
"id05272": "m",
"id05273": "f",
"id05274": "f",
"id05275": "f",
"id05276": "f",
"id05277": "f",
"id05278": "f",
"id05280": "f",
"id05282": "m",
"id05283": "f",
"id05284": "f",
"id05286": "m",
"id05287": "m",
"id05288": "m",
"id05289": "m",
"id05290": "m",
"id05291": "m",
"id05292": "m",
"id05293": "m",
"id05295": "m",
"id05296": "m",
"id05297": "m",
"id05300": "m",
"id05301": "m",
"id05303": "f",
"id05306": "f",
"id05308": "f",
"id05310": "m",
"id05312": "m",
"id05313": "f",
"id05314": "f",
"id05315": "f",
"id05316": "f",
"id05317": "m",
"id05319": "f",
"id05321": "m",
"id05323": "m",
"id05324": "m",
"id05328": "m",
"id05332": "m",
"id05333": "m",
"id05334": "m",
"id05335": "m",
"id05339": "f",
"id05341": "m",
"id05342": "m",
"id05343": "m",
"id05344": "m",
"id05345": "m",
"id05346": "m",
"id05348": "m",
"id05349": "m",
"id05350": "m",
"id05351": "m",
"id05352": "m",
"id05353": "m",
"id05354": "m",
"id05357": "m",
"id05358": "m",
"id05369": "f",
"id05370": "f",
"id05371": "f",
"id05372": "f",
"id05373": "f",
"id05375": "m",
"id05376": "f",
"id05378": "m",
"id05379": "f",
"id05380": "m",
"id05381": "m",
"id05383": "m",
"id05384": "m",
"id05387": "m",
"id05388": "f",
"id05392": "f",
"id05398": "f",
"id05399": "f",
"id05401": "f",
"id05402": "f",
"id05403": "f",
"id05404": "f",
"id05405": "f",
"id05408": "m",
"id05421": "f",
"id05422": "f",
"id05423": "f",
"id05424": "f",
"id05426": "m",
"id05427": "m",
"id05428": "m",
"id05429": "m",
"id05430": "m",
"id05431": "m",
"id05432": "f",
"id05433": "m",
"id05434": "f",
"id05435": "f",
"id05438": "f",
"id05441": "f",
"id05442": "f",
"id05443": "f",
"id05444": "f",
"id05445": "m",
"id05448": "m",
"id05449": "f",
"id05450": "f",
"id05451": "m",
"id05453": "m",
"id05454": "m",
"id05456": "f",
"id05457": "f",
"id05459": "f",
"id05460": "m",
"id05465": "m",
"id05466": "m",
"id05467": "f",
"id05468": "f",
"id05469": "f",
"id05470": "f",
"id05471": "m",
"id05472": "m",
"id05474": "f",
"id05476": "m",
"id05477": "f",
"id05478": "f",
"id05479": "m",
"id05480": "m",
"id05481": "m",
"id05483": "f",
"id05486": "m",
"id05487": "m",
"id05490": "m",
"id05491": "m",
"id05492": "m",
"id05493": "m",
"id05494": "m",
"id05496": "f",
"id05502": "m",
"id05503": "m",
"id05504": "m",
"id05505": "m",
"id05506": "m",
"id05507": "m",
"id05508": "m",
"id05509": "m",
"id05510": "m",
"id05512": "m",
"id05513": "m",
"id05515": "m",
"id05516": "m",
"id05517": "m",
"id05518": "m",
"id05519": "m",
"id05520": "m",
"id05521": "m",
"id05522": "f",
"id05524": "m",
"id05526": "m",
"id05527": "m",
"id05528": "f",
"id05529": "m",
"id05534": "m",
"id05535": "m",
"id05536": "m",
"id05538": "m",
"id05541": "m",
"id05545": "m",
"id05546": "m",
"id05547": "m",
"id05549": "m",
"id05551": "m",
"id05552": "m",
"id05553": "m",
"id05554": "m",
"id05555": "m",
"id05556": "m",
"id05557": "m",
"id05558": "m",
"id05559": "m",
"id05561": "m",
"id05562": "m",
"id05563": "m",
"id05564": "m",
"id05567": "f",
"id05568": "m",
"id05570": "m",
"id05573": "f",
"id05574": "f",
"id05575": "f",
"id05576": "f",
"id05577": "f",
"id05579": "f",
"id05581": "f",
"id05583": "f",
"id05584": "f",
"id05585": "f",
"id05586": "m",
"id05588": "f",
"id05591": "f",
"id05594": "f",
"id05607": "m",
"id05609": "f",
"id05610": "f",
"id05612": "f",
"id05613": "f",
"id05614": "f",
"id05615": "f",
"id05616": "f",
"id05619": "f",
"id05620": "f",
"id05621": "f",
"id05622": "f",
"id05623": "f",
"id05624": "f",
"id05625": "f",
"id05627": "f",
"id05628": "f",
"id05631": "f",
"id05632": "f",
"id05633": "f",
"id05634": "f",
"id05639": "m",
"id05640": "f",
"id05641": "m",
"id05642": "f",
"id05643": "f",
"id05644": "f",
"id05645": "m",
"id05647": "m",
"id05648": "m",
"id05649": "m",
"id05654": "m",
"id05657": "m",
"id05658": "m",
"id05660": "m",
"id05663": "f",
"id05664": "f",
"id05665": "f",
"id05666": "f",
"id05667": "f",
"id05668": "f",
"id05670": "f",
"id05671": "m",
"id05672": "m",
"id05677": "f",
"id05678": "m",
"id05681": "m",
"id05684": "m",
"id05685": "m",
"id05686": "m",
"id05687": "m",
"id05688": "m",
"id05689": "m",
"id05690": "m",
"id05691": "m",
"id05692": "m",
"id05693": "m",
"id05694": "m",
"id05695": "m",
"id05696": "m",
"id05697": "m",
"id05698": "m",
"id05699": "m",
"id05700": "m",
"id05703": "m",
"id05704": "m",
"id05706": "f",
"id05707": "f",
"id05708": "f",
"id05710": "f",
"id05714": "m",
"id05715": "m",
"id05716": "f",
"id05718": "f",
"id05719": "f",
"id05721": "f",
"id05722": "f",
"id05723": "f",
"id05724": "f",
"id05725": "f",
"id05726": "m",
"id05727": "f",
"id05728": "m",
"id05729": "m",
"id05731": "m",
"id05733": "m",
"id05734": "m",
"id05735": "m",
"id05737": "m",
"id05738": "m",
"id05739": "m",
"id05740": "m",
"id05741": "m",
"id05742": "m",
"id05743": "m",
"id05745": "f",
"id05746": "f",
"id05747": "f",
"id05748": "f",
"id05749": "m",
"id05750": "f",
"id05751": "m",
"id05752": "f",
"id05754": "m",
"id05756": "m",
"id05757": "f",
"id05758": "m",
"id05759": "m",
"id05760": "f",
"id05762": "f",
"id05764": "f",
"id05765": "f",
"id05766": "f",
"id05767": "f",
"id05768": "f",
"id05769": "f",
"id05770": "f",
"id05771": "f",
"id05772": "f",
"id05774": "m",
"id05776": "f",
"id05777": "f",
"id05779": "f",
"id05783": "f",
"id05784": "f",
"id05788": "m",
"id05792": "m",
"id05793": "f",
"id05796": "m",
"id05797": "m",
"id05798": "m",
"id05799": "m",
"id05801": "f",
"id05804": "m",
"id05805": "m",
"id05806": "f",
"id05808": "m",
"id05810": "m",
"id05811": "m",
"id05812": "m",
"id05813": "m",
"id05814": "m",
"id05815": "m",
"id05816": "m",
"id05827": "m",
"id05828": "m",
"id05829": "m",
"id05830": "m",
"id05831": "m",
"id05832": "m",
"id05833": "m",
"id05834": "m",
"id05835": "m",
"id05836": "m",
"id05837": "m",
"id05838": "m",
"id05841": "m",
"id05843": "f",
"id05844": "f",
"id05845": "f",
"id05847": "f",
"id05849": "f",
"id05850": "m",
"id05851": "m",
"id05852": "m",
"id05853": "m",
"id05856": "m",
"id05858": "m",
"id05860": "m",
"id05861": "m",
"id05864": "m",
"id05868": "m",
"id05869": "m",
"id05870": "m",
"id05871": "m",
"id05873": "m",
"id05874": "m",
"id05875": "m",
"id05876": "m",
"id05877": "m",
"id05878": "m",
"id05879": "m",
"id05880": "m",
"id05881": "m",
"id05882": "m",
"id05887": "m",
"id05890": "m",
"id05891": "f",
"id05892": "f",
"id05893": "f",
"id05894": "m",
"id05895": "f",
"id05904": "m",
"id05905": "f",
"id05906": "f",
"id05907": "f",
"id05908": "f",
"id05913": "f",
"id05914": "f",
"id05915": "f",
"id05916": "f",
"id05917": "f",
"id05919": "f",
"id05920": "f",
"id05921": "f",
"id05922": "m",
"id05923": "m",
"id05926": "m",
"id05927": "m",
"id05930": "m",
"id05931": "f",
"id05932": "f",
"id05934": "m",
"id05935": "f",
"id05936": "f",
"id05938": "f",
"id05939": "m",
"id05940": "m",
"id05942": "f",
"id05943": "f",
"id05944": "f",
"id05945": "f",
"id05946": "f",
"id05951": "f",
"id05952": "f",
"id05953": "f",
"id05954": "f",
"id05956": "f",
"id05957": "f",
"id05958": "f",
"id05959": "f",
"id05960": "m",
"id05961": "m",
"id05962": "f",
"id05963": "m",
"id05964": "f",
"id05966": "f",
"id05969": "m",
"id05970": "f",
"id05976": "m",
"id05977": "m",
"id05978": "m",
"id05980": "f",
"id05981": "f",
"id05982": "m",
"id05983": "m",
"id05984": "m",
"id05985": "m",
"id05986": "m",
"id05987": "m",
"id05988": "m",
"id05989": "m",
"id05990": "m",
"id05991": "m",
"id05993": "m",
"id05994": "m",
"id05996": "m",
"id05997": "m",
"id05998": "m",
"id05999": "m",
"id06000": "m",
"id06002": "m",
"id06003": "m",
"id06004": "m",
"id06005": "m",
"id06006": "m",
"id06007": "m",
"id06008": "m",
"id06010": "m",
"id06011": "f",
"id06012": "m",
"id06013": "m",
"id06014": "m",
"id06015": "m",
"id06016": "m",
"id06017": "m",
"id06018": "m",
"id06019": "m",
"id06020": "m",
"id06021": "m",
"id06022": "m",
"id06023": "m",
"id06024": "m",
"id06025": "m",
"id06026": "f",
"id06027": "f",
"id06028": "f",
"id06029": "f",
"id06038": "m",
"id06040": "m",
"id06042": "m",
"id06044": "m",
"id06045": "m",
"id06046": "m",
"id06049": "m",
"id06050": "m",
"id06051": "f",
"id06052": "f",
"id06054": "f",
"id06055": "f",
"id06056": "f",
"id06058": "f",
"id06059": "f",
"id06060": "f",
"id06061": "f",
"id06062": "f",
"id06064": "f",
"id06065": "f",
"id06066": "f",
"id06067": "m",
"id06068": "m",
"id06070": "m",
"id06071": "m",
"id06073": "m",
"id06081": "f",
"id06084": "m",
"id06086": "m",
"id06087": "m",
"id06092": "m",
"id06095": "m",
"id06096": "f",
"id06097": "m",
"id06103": "m",
"id06104": "m",
"id06105": "m",
"id06106": "m",
"id06107": "m",
"id06108": "m",
"id06109": "m",
"id06110": "m",
"id06111": "m",
"id06112": "m",
"id06113": "m",
"id06114": "m",
"id06115": "m",
"id06116": "m",
"id06117": "m",
"id06118": "m",
"id06119": "m",
"id06120": "m",
"id06121": "m",
"id06122": "m",
"id06124": "m",
"id06128": "m",
"id06130": "f",
"id06131": "m",
"id06133": "m",
"id06134": "f",
"id06135": "m",
"id06138": "m",
"id06139": "m",
"id06140": "m",
"id06142": "m",
"id06143": "m",
"id06145": "f",
"id06146": "f",
"id06148": "f",
"id06152": "m",
"id06154": "f",
"id06155": "f",
"id06156": "f",
"id06158": "f",
"id06159": "f",
"id06161": "f",
"id06162": "m",
"id06163": "m",
"id06164": "m",
"id06165": "f",
"id06166": "f",
"id06168": "f",
"id06172": "f",
"id06173": "f",
"id06174": "f",
"id06176": "f",
"id06178": "m",
"id06183": "m",
"id06184": "m",
"id06185": "m",
"id06188": "f",
"id06189": "m",
"id06191": "m",
"id06194": "f",
"id06195": "f",
"id06196": "m",
"id06197": "m",
"id06199": "m",
"id06200": "f",
"id06202": "m",
"id06203": "m",
"id06206": "m",
"id06207": "f",
"id06209": "m",
"id06210": "m",
"id06211": "m",
"id06212": "m",
"id06213": "m",
"id06214": "m",
"id06215": "m",
"id06216": "m",
"id06217": "m",
"id06218": "m",
"id06220": "f",
"id06221": "f",
"id06222": "f",
"id06225": "f",
"id06226": "f",
"id06227": "f",
"id06228": "f",
"id06229": "f",
"id06231": "f",
"id06232": "f",
"id06233": "f",
"id06234": "f",
"id06235": "f",
"id06236": "f",
"id06237": "f",
"id06238": "f",
"id06239": "f",
"id06242": "f",
"id06244": "m",
"id06245": "m",
"id06246": "m",
"id06247": "f",
"id06248": "m",
"id06249": "f",
"id06250": "m",
"id06251": "m",
"id06252": "m",
"id06254": "f",
"id06255": "m",
"id06256": "m",
"id06258": "m",
"id06261": "f",
"id06262": "m",
"id06263": "m",
"id06264": "m",
"id06265": "m",
"id06267": "m",
"id06268": "f",
"id06269": "m",
"id06270": "m",
"id06274": "m",
"id06275": "m",
"id06276": "m",
"id06277": "f",
"id06278": "f",
"id06279": "m",
"id06280": "m",
"id06288": "f",
"id06289": "m",
"id06292": "f",
"id06293": "f",
"id06295": "f",
"id06297": "f",
"id06298": "f",
"id06299": "f",
"id06301": "f",
"id06302": "f",
"id06303": "m",
"id06304": "f",
"id06307": "f",
"id06310": "f",
"id06311": "m",
"id06319": "f",
"id06320": "f",
"id06321": "f",
"id06322": "f",
"id06323": "f",
"id06324": "f",
"id06328": "f",
"id06331": "f",
"id06332": "f",
"id06333": "f",
"id06334": "m",
"id06335": "m",
"id06336": "m",
"id06338": "f",
"id06339": "m",
"id06340": "m",
"id06341": "m",
"id06342": "f",
"id06343": "f",
"id06344": "f",
"id06346": "f",
"id06347": "f",
"id06349": "f",
"id06350": "f",
"id06351": "f",
"id06353": "m",
"id06354": "m",
"id06355": "m",
"id06356": "m",
"id06357": "f",
"id06358": "f",
"id06360": "f",
"id06361": "f",
"id06362": "m",
"id06364": "m",
"id06368": "m",
"id06370": "f",
"id06371": "f",
"id06375": "f",
"id06376": "f",
"id06377": "f",
"id06378": "f",
"id06379": "f",
"id06380": "f",
"id06381": "f",
"id06382": "f",
"id06383": "f",
"id06384": "f",
"id06385": "f",
"id06386": "f",
"id06388": "f",
"id06389": "f",
"id06390": "f",
"id06392": "f",
"id06393": "f",
"id06394": "f",
"id06395": "f",
"id06396": "f",
"id06397": "f",
"id06398": "f",
"id06399": "f",
"id06400": "f",
"id06403": "m",
"id06404": "m",
"id06405": "m",
"id06406": "m",
"id06408": "f",
"id06409": "f",
"id06411": "f",
"id06412": "m",
"id06413": "m",
"id06414": "m",
"id06415": "m",
"id06416": "m",
"id06417": "m",
"id06418": "m",
"id06422": "f",
"id06423": "m",
"id06424": "f",
"id06425": "f",
"id06426": "f",
"id06427": "f",
"id06428": "f",
"id06429": "f",
"id06430": "m",
"id06431": "f",
"id06432": "m",
"id06434": "f",
"id06435": "m",
"id06436": "f",
"id06437": "f",
"id06438": "f",
"id06439": "f",
"id06440": "f",
"id06443": "f",
"id06445": "f",
"id06446": "m",
"id06447": "m",
"id06448": "m",
"id06449": "m",
"id06450": "m",
"id06453": "m",
"id06454": "m",
"id06456": "m",
"id06457": "m",
"id06459": "m",
"id06462": "f",
"id06465": "m",
"id06467": "m",
"id06468": "f",
"id06469": "f",
"id06470": "m",
"id06471": "m",
"id06472": "m",
"id06474": "m",
"id06476": "m",
"id06477": "m",
"id06478": "m",
"id06479": "m",
"id06480": "m",
"id06481": "m",
"id06482": "m",
"id06483": "m",
"id06484": "m",
"id06485": "m",
"id06486": "m",
"id06487": "m",
"id06489": "m",
"id06490": "m",
"id06491": "m",
"id06493": "m",
"id06494": "f",
"id06495": "m",
"id06496": "m",
"id06497": "m",
"id06498": "m",
"id06499": "m",
"id06500": "m",
"id06502": "f",
"id06503": "m",
"id06504": "f",
"id06505": "m",
"id06506": "f",
"id06507": "f",
"id06509": "m",
"id06511": "m",
"id06516": "m",
"id06517": "f",
"id06518": "f",
"id06519": "f",
"id06520": "f",
"id06524": "f",
"id06526": "f",
"id06528": "f",
"id06529": "f",
"id06530": "m",
"id06531": "m",
"id06535": "m",
"id06536": "m",
"id06537": "m",
"id06538": "m",
"id06539": "m",
"id06541": "f",
"id06544": "m",
"id06547": "f",
"id06548": "f",
"id06550": "f",
"id06551": "m",
"id06552": "f",
"id06553": "m",
"id06554": "f",
"id06555": "m",
"id06556": "m",
"id06557": "m",
"id06559": "m",
"id06560": "m",
"id06574": "f",
"id06575": "f",
"id06576": "f",
"id06577": "f",
"id06578": "f",
"id06581": "f",
"id06582": "f",
"id06583": "f",
"id06584": "f",
"id06586": "m",
"id06588": "m",
"id06589": "m",
"id06591": "m",
"id06593": "m",
"id06594": "m",
"id06595": "m",
"id06596": "m",
"id06597": "m",
"id06598": "m",
"id06599": "f",
"id06607": "m",
"id06608": "f",
"id06609": "f",
"id06612": "f",
"id06613": "m",
"id06614": "m",
"id06616": "m",
"id06617": "m",
"id06619": "f",
"id06621": "m",
"id06622": "m",
"id06624": "f",
"id06625": "f",
"id06631": "m",
"id06633": "m",
"id06637": "m",
"id06640": "f",
"id06642": "m",
"id06643": "m",
"id06645": "m",
"id06648": "m",
"id06649": "m",
"id06650": "m",
"id06651": "m",
"id06661": "m",
"id06662": "m",
"id06663": "m",
"id06672": "m",
"id06673": "m",
"id06674": "m",
"id06676": "m",
"id06683": "m",
"id06684": "f",
"id06685": "m",
"id06686": "m",
"id06687": "m",
"id06688": "m",
"id06689": "m",
"id06691": "f",
"id06692": "f",
"id06694": "m",
"id06695": "m",
"id06696": "m",
"id06697": "f",
"id06698": "f",
"id06701": "m",
"id06702": "m",
"id06703": "m",
"id06704": "m",
"id06706": "m",
"id06708": "m",
"id06709": "m",
"id06713": "f",
"id06714": "m",
"id06715": "m",
"id06716": "m",
"id06717": "m",
"id06721": "m",
"id06722": "f",
"id06723": "m",
"id06724": "f",
"id06725": "f",
"id06726": "m",
"id06730": "m",
"id06732": "m",
"id06741": "m",
"id06743": "f",
"id06744": "f",
"id06745": "f",
"id06747": "f",
"id06748": "f",
"id06749": "f",
"id06750": "f",
"id06752": "f",
"id06753": "m",
"id06754": "f",
"id06757": "f",
"id06758": "f",
"id06759": "f",
"id06760": "m",
"id06761": "m",
"id06763": "m",
"id06765": "m",
"id06767": "m",
"id06768": "m",
"id06771": "m",
"id06772": "f",
"id06775": "f",
"id06776": "m",
"id06779": "m",
"id06782": "f",
"id06784": "m",
"id06791": "m",
"id06792": "m",
"id06793": "m",
"id06794": "m",
"id06795": "m",
"id06797": "m",
"id06798": "m",
"id06799": "m",
"id06807": "m",
"id06809": "m",
"id06810": "m",
"id06811": "m",
"id06812": "m",
"id06813": "m",
"id06815": "m",
"id06816": "m",
"id06817": "m",
"id06818": "m",
"id06819": "m",
"id06820": "m",
"id06821": "m",
"id06822": "m",
"id06823": "f",
"id06828": "f",
"id06829": "f",
"id06830": "m",
"id06833": "m",
"id06835": "m",
"id06836": "m",
"id06837": "m",
"id06838": "m",
"id06839": "m",
"id06840": "m",
"id06841": "m",
"id06842": "m",
"id06843": "m",
"id06844": "m",
"id06845": "m",
"id06846": "m",
"id06847": "m",
"id06848": "m",
"id06849": "m",
"id06850": "m",
"id06851": "m",
"id06852": "m",
"id06853": "m",
"id06854": "m",
"id06855": "m",
"id06856": "m",
"id06857": "m",
"id06858": "m",
"id06860": "m",
"id06861": "f",
"id06862": "f",
"id06863": "f",
"id06864": "f",
"id06866": "f",
"id06867": "f",
"id06869": "f",
"id06870": "m",
"id06874": "f",
"id06875": "f",
"id06876": "f",
"id06878": "m",
"id06879": "m",
"id06881": "m",
"id06883": "m",
"id06887": "m",
"id06888": "m",
"id06889": "m",
"id06890": "m",
"id06892": "m",
"id06893": "f",
"id06896": "f",
"id06900": "m",
"id06905": "m",
"id06906": "f",
"id06907": "f",
"id06908": "f",
"id06909": "m",
"id06910": "m",
"id06911": "f",
"id06912": "f",
"id06913": "m",
"id06914": "f",
"id06915": "f",
"id06916": "f",
"id06918": "m",
"id06920": "m",
"id06922": "m",
"id06926": "f",
"id06927": "f",
"id06928": "m",
"id06929": "m",
"id06930": "f",
"id06931": "m",
"id06932": "f",
"id06934": "m",
"id06935": "m",
"id06936": "m",
"id06937": "m",
"id06938": "m",
"id06939": "m",
"id06940": "m",
"id06941": "m",
"id06942": "m",
"id06944": "m",
"id06945": "m",
"id06946": "m",
"id06948": "m",
"id06949": "m",
"id06950": "m",
"id06951": "m",
"id06952": "m",
"id06953": "m",
"id06954": "m",
"id06957": "m",
"id06958": "m",
"id06960": "f",
"id06961": "m",
"id06962": "m",
"id06963": "m",
"id06964": "m",
"id06966": "m",
"id06967": "m",
"id06968": "m",
"id06969": "m",
"id06970": "m",
"id06971": "m",
"id06972": "m",
"id06973": "m",
"id06974": "m",
"id06976": "m",
"id06982": "m",
"id06983": "m",
"id06984": "m",
"id06985": "m",
"id06986": "m",
"id06987": "m",
"id06988": "m",
"id06989": "m",
"id06990": "m",
"id06991": "m",
"id06994": "m",
"id06995": "m",
"id06996": "m",
"id06997": "m",
"id06998": "m",
"id06999": "f",
"id07001": "f",
"id07002": "f",
"id07003": "f",
"id07004": "f",
"id07005": "f",
"id07007": "f",
"id07008": "f",
"id07009": "f",
"id07010": "m",
"id07011": "m",
"id07012": "m",
"id07015": "m",
"id07017": "m",
"id07019": "m",
"id07020": "m",
"id07022": "m",
"id07026": "f",
"id07031": "f",
"id07032": "f",
"id07034": "f",
"id07036": "f",
"id07037": "m",
"id07039": "f",
"id07042": "m",
"id07043": "m",
"id07044": "f",
"id07045": "f",
"id07046": "f",
"id07048": "f",
"id07049": "f",
"id07050": "f",
"id07051": "f",
"id07052": "f",
"id07053": "f",
"id07054": "m",
"id07055": "f",
"id07056": "f",
"id07057": "m",
"id07058": "m",
"id07059": "m",
"id07060": "m",
"id07062": "m",
"id07063": "f",
"id07064": "f",
"id07065": "f",
"id07066": "f",
"id07071": "m",
"id07072": "f",
"id07074": "f",
"id07076": "f",
"id07077": "m",
"id07078": "f",
"id07080": "f",
"id07084": "m",
"id07085": "m",
"id07086": "m",
"id07087": "f",
"id07088": "m",
"id07089": "m",
"id07092": "f",
"id07093": "f",
"id07094": "f",
"id07095": "f",
"id07097": "f",
"id07098": "f",
"id07099": "f",
"id07100": "m",
"id07102": "m",
"id07103": "m",
"id07104": "m",
"id07105": "m",
"id07107": "m",
"id07108": "m",
"id07109": "m",
"id07112": "f",
"id07113": "f",
"id07114": "f",
"id07115": "f",
"id07116": "f",
"id07117": "f",
"id07118": "f",
"id07120": "f",
"id07121": "f",
"id07122": "m",
"id07123": "f",
"id07124": "f",
"id07128": "f",
"id07130": "f",
"id07131": "m",
"id07132": "m",
"id07133": "f",
"id07134": "m",
"id07135": "f",
"id07136": "f",
"id07137": "m",
"id07140": "m",
"id07145": "m",
"id07146": "m",
"id07148": "m",
"id07151": "m",
"id07153": "m",
"id07154": "m",
"id07155": "m",
"id07158": "f",
"id07160": "m",
"id07161": "m",
"id07162": "f",
"id07163": "m",
"id07164": "m",
"id07165": "m",
"id07166": "m",
"id07167": "m",
"id07168": "f",
"id07169": "f",
"id07170": "m",
"id07171": "m",
"id07173": "m",
"id07175": "m",
"id07176": "m",
"id07177": "m",
"id07178": "m",
"id07179": "m",
"id07180": "f",
"id07181": "m",
"id07182": "m",
"id07183": "f",
"id07185": "m",
"id07186": "f",
"id07187": "f",
"id07188": "m",
"id07189": "m",
"id07191": "m",
"id07192": "m",
"id07194": "m",
"id07195": "m",
"id07196": "m",
"id07197": "f",
"id07198": "m",
"id07199": "m",
"id07200": "m",
"id07202": "m",
"id07204": "f",
"id07205": "m",
"id07206": "m",
"id07207": "m",
"id07208": "f",
"id07209": "f",
"id07210": "m",
"id07212": "m",
"id07213": "m",
"id07214": "m",
"id07215": "m",
"id07217": "m",
"id07218": "f",
"id07219": "m",
"id07220": "m",
"id07221": "m",
"id07223": "m",
"id07227": "f",
"id07228": "m",
"id07229": "m",
"id07230": "f",
"id07232": "f",
"id07233": "m",
"id07234": "f",
"id07235": "m",
"id07236": "f",
"id07238": "f",
"id07240": "m",
"id07241": "m",
"id07242": "f",
"id07243": "m",
"id07244": "m",
"id07246": "m",
"id07247": "m",
"id07250": "m",
"id07251": "m",
"id07253": "f",
"id07254": "f",
"id07255": "f",
"id07256": "f",
"id07258": "f",
"id07259": "f",
"id07262": "m",
"id07263": "m",
"id07264": "m",
"id07265": "m",
"id07268": "f",
"id07269": "f",
"id07272": "m",
"id07273": "m",
"id07275": "m",
"id07276": "f",
"id07277": "f",
"id07278": "f",
"id07279": "m",
"id07283": "f",
"id07284": "f",
"id07285": "f",
"id07288": "m",
"id07290": "m",
"id07292": "f",
"id07293": "f",
"id07294": "f",
"id07295": "f",
"id07296": "m",
"id07297": "m",
"id07299": "f",
"id07303": "m",
"id07305": "m",
"id07306": "m",
"id07308": "f",
"id07311": "m",
"id07312": "m",
"id07314": "m",
"id07316": "m",
"id07320": "m",
"id07321": "m",
"id07322": "m",
"id07332": "m",
"id07333": "m",
"id07334": "m",
"id07335": "m",
"id07337": "m",
"id07338": "m",
"id07341": "m",
"id07342": "m",
"id07343": "m",
"id07344": "m",
"id07345": "m",
"id07346": "m",
"id07349": "m",
"id07350": "m",
"id07351": "m",
"id07352": "m",
"id07353": "m",
"id07354": "m",
"id07355": "m",
"id07356": "m",
"id07357": "m",
"id07358": "m",
"id07359": "m",
"id07360": "m",
"id07361": "m",
"id07362": "f",
"id07363": "m",
"id07366": "m",
"id07367": "m",
"id07368": "m",
"id07369": "m",
"id07370": "m",
"id07371": "m",
"id07374": "f",
"id07375": "m",
"id07376": "f",
"id07377": "f",
"id07383": "f",
"id07384": "m",
"id07385": "m",
"id07388": "f",
"id07389": "f",
"id07392": "m",
"id07393": "m",
"id07394": "f",
"id07395": "m",
"id07396": "m",
"id07397": "m",
"id07398": "m",
"id07399": "m",
"id07400": "m",
"id07401": "m",
"id07402": "m",
"id07403": "m",
"id07404": "m",
"id07405": "m",
"id07407": "m",
"id07408": "m",
"id07409": "m",
"id07411": "m",
"id07412": "m",
"id07413": "m",
"id07414": "m",
"id07415": "m",
"id07416": "m",
"id07417": "m",
"id07418": "m",
"id07419": "m",
"id07421": "m",
"id07422": "m",
"id07423": "m",
"id07426": "m",
"id07427": "m",
"id07428": "m",
"id07429": "m",
"id07430": "m",
"id07431": "m",
"id07432": "m",
"id07433": "m",
"id07434": "m",
"id07437": "m",
"id07438": "m",
"id07439": "m",
"id07441": "m",
"id07442": "m",
"id07445": "m",
"id07446": "m",
"id07447": "m",
"id07448": "m",
"id07452": "m",
"id07453": "m",
"id07454": "m",
"id07455": "m",
"id07456": "m",
"id07457": "f",
"id07459": "f",
"id07460": "f",
"id07461": "m",
"id07462": "f",
"id07463": "m",
"id07465": "f",
"id07466": "m",
"id07468": "m",
"id07469": "m",
"id07470": "m",
"id07471": "f",
"id07473": "m",
"id07474": "f",
"id07477": "m",
"id07478": "m",
"id07483": "m",
"id07484": "m",
"id07489": "m",
"id07491": "m",
"id07492": "m",
"id07493": "m",
"id07494": "m",
"id07495": "m",
"id07496": "m",
"id07497": "m",
"id07498": "m",
"id07500": "m",
"id07502": "m",
"id07506": "m",
"id07507": "m",
"id07508": "f",
"id07510": "m",
"id07511": "m",
"id07512": "m",
"id07517": "m",
"id07519": "m",
"id07520": "m",
"id07522": "f",
"id07524": "f",
"id07525": "m",
"id07526": "m",
"id07527": "m",
"id07528": "m",
"id07529": "m",
"id07531": "m",
"id07532": "m",
"id07533": "m",
"id07534": "f",
"id07535": "m",
"id07536": "m",
"id07537": "m",
"id07538": "m",
"id07539": "f",
"id07540": "m",
"id07541": "m",
"id07543": "m",
"id07545": "m",
"id07546": "m",
"id07547": "m",
"id07548": "m",
"id07549": "m",
"id07550": "m",
"id07551": "m",
"id07552": "m",
"id07553": "m",
"id07554": "m",
"id07555": "m",
"id07556": "m",
"id07558": "f",
"id07560": "f",
"id07561": "f",
"id07562": "f",
"id07564": "f",
"id07565": "f",
"id07566": "f",
"id07567": "f",
"id07568": "f",
"id07569": "f",
"id07570": "f",
"id07571": "f",
"id07572": "f",
"id07574": "f",
"id07575": "f",
"id07576": "m",
"id07577": "m",
"id07578": "m",
"id07579": "m",
"id07580": "f",
"id07581": "f",
"id07582": "m",
"id07583": "f",
"id07585": "m",
"id07586": "m",
"id07587": "m",
"id07588": "m",
"id07589": "m",
"id07590": "m",
"id07591": "f",
"id07592": "f",
"id07593": "f",
"id07598": "f",
"id07599": "f",
"id07600": "f",
"id07611": "f",
"id07612": "f",
"id07613": "f",
"id07614": "m",
"id07615": "f",
"id07616": "m",
"id07617": "m",
"id07618": "m",
"id07620": "m",
"id07621": "m",
"id07622": "m",
"id07623": "m",
"id07624": "m",
"id07625": "m",
"id07626": "m",
"id07627": "m",
"id07628": "m",
"id07630": "f",
"id07631": "f",
"id07635": "m",
"id07636": "m",
"id07637": "m",
"id07638": "m",
"id07639": "m",
"id07640": "m",
"id07641": "m",
"id07642": "m",
"id07643": "m",
"id07644": "m",
"id07646": "f",
"id07647": "m",
"id07648": "m",
"id07649": "m",
"id07652": "m",
"id07656": "m",
"id07658": "m",
"id07660": "m",
"id07661": "f",
"id07662": "m",
"id07663": "f",
"id07664": "m",
"id07666": "m",
"id07669": "f",
"id07670": "f",
"id07671": "f",
"id07672": "f",
"id07673": "m",
"id07675": "f",
"id07678": "f",
"id07679": "m",
"id07680": "f",
"id07681": "m",
"id07682": "m",
"id07683": "f",
"id07685": "f",
"id07686": "f",
"id07688": "f",
"id07689": "f",
"id07691": "m",
"id07692": "f",
"id07693": "f",
"id07694": "f",
"id07695": "f",
"id07696": "m",
"id07697": "f",
"id07700": "m",
"id07701": "m",
"id07702": "m",
"id07703": "m",
"id07704": "m",
"id07705": "m",
"id07706": "m",
"id07707": "m",
"id07708": "m",
"id07711": "m",
"id07713": "m",
"id07714": "m",
"id07715": "m",
"id07716": "m",
"id07717": "m",
"id07719": "m",
"id07720": "f",
"id07721": "f",
"id07723": "f",
"id07724": "f",
"id07725": "f",
"id07726": "f",
"id07727": "f",
"id07728": "f",
"id07730": "m",
"id07732": "m",
"id07733": "m",
"id07734": "f",
"id07735": "m",
"id07736": "f",
"id07737": "m",
"id07738": "m",
"id07739": "m",
"id07740": "m",
"id07741": "m",
"id07742": "f",
"id07744": "m",
"id07745": "m",
"id07747": "m",
"id07748": "f",
"id07749": "m",
"id07750": "f",
"id07751": "f",
"id07752": "f",
"id07753": "f",
"id07755": "f",
"id07756": "f",
"id07758": "f",
"id07759": "f",
"id07760": "f",
"id07761": "m",
"id07764": "f",
"id07768": "m",
"id07769": "f",
"id07770": "f",
"id07771": "f",
"id07773": "m",
"id07774": "m",
"id07775": "m",
"id07776": "f",
"id07780": "f",
"id07783": "f",
"id07785": "f",
"id07786": "f",
"id07798": "f",
"id07799": "f",
"id07800": "m",
"id07801": "m",
"id07802": "f",
"id07803": "f",
"id07805": "m",
"id07807": "m",
"id07809": "f",
"id07811": "m",
"id07812": "m",
"id07814": "f",
"id07816": "f",
"id07818": "f",
"id07819": "m",
"id07820": "m",
"id07824": "m",
"id07825": "m",
"id07826": "m",
"id07827": "m",
"id07828": "m",
"id07829": "m",
"id07831": "m",
"id07832": "m",
"id07833": "m",
"id07834": "m",
"id07835": "m",
"id07837": "m",
"id07838": "m",
"id07839": "f",
"id07840": "m",
"id07841": "m",
"id07844": "m",
"id07845": "m",
"id07846": "m",
"id07847": "m",
"id07848": "m",
"id07849": "m",
"id07850": "m",
"id07851": "m",
"id07852": "m",
"id07853": "m",
"id07856": "m",
"id07857": "m",
"id07858": "m",
"id07860": "m",
"id07861": "m",
"id07862": "m",
"id07863": "m",
"id07864": "m",
"id07866": "m",
"id07868": "m",
"id07869": "m",
"id07870": "m",
"id07874": "m",
"id07875": "m",
"id07876": "m",
"id07882": "m",
"id07883": "m",
"id07884": "f",
"id07885": "m",
"id07886": "m",
"id07887": "f",
"id07894": "m",
"id07895": "m",
"id07898": "m",
"id07901": "f",
"id07902": "m",
"id07910": "f",
"id07911": "f",
"id07912": "f",
"id07914": "m",
"id07916": "m",
"id07917": "m",
"id07926": "m",
"id07927": "m",
"id07928": "m",
"id07941": "m",
"id07943": "m",
"id07944": "m",
"id07947": "f",
"id07948": "m",
"id07949": "f",
"id07950": "f",
"id07951": "m",
"id07953": "f",
"id07954": "f",
"id07955": "f",
"id07956": "f",
"id07957": "f",
"id07958": "m",
"id07959": "m",
"id07960": "m",
"id07961": "m",
"id07962": "m",
"id07963": "m",
"id07964": "m",
"id07967": "m",
"id07968": "m",
"id07970": "m",
"id07971": "f",
"id07972": "f",
"id07973": "m",
"id07974": "m",
"id07975": "f",
"id07976": "f",
"id07977": "m",
"id07978": "m",
"id07979": "m",
"id07980": "f",
"id07981": "f",
"id07982": "f",
"id07983": "f",
"id07984": "m",
"id07985": "m",
"id07987": "m",
"id07988": "m",
"id07989": "m",
"id07990": "m",
"id07991": "f",
"id07992": "m",
"id07993": "m",
"id07995": "m",
"id07996": "f",
"id07997": "m",
"id07998": "m",
"id07999": "m",
"id08000": "m",
"id08001": "m",
"id08002": "m",
"id08003": "f",
"id08012": "f",
"id08013": "f",
"id08014": "f",
"id08015": "f",
"id08016": "f",
"id08018": "f",
"id08019": "f",
"id08020": "f",
"id08022": "f",
"id08023": "m",
"id08024": "f",
"id08025": "m",
"id08026": "f",
"id08027": "f",
"id08028": "f",
"id08029": "m",
"id08030": "m",
"id08037": "m",
"id08039": "f",
"id08040": "m",
"id08041": "f",
"id08042": "f",
"id08043": "f",
"id08044": "f",
"id08046": "f",
"id08047": "f",
"id08048": "m",
"id08049": "f",
"id08052": "f",
"id08053": "f",
"id08054": "f",
"id08064": "m",
"id08065": "m",
"id08066": "f",
"id08067": "m",
"id08068": "m",
"id08071": "f",
"id08073": "f",
"id08074": "f",
"id08078": "m",
"id08079": "m",
"id08080": "m",
"id08081": "m",
"id08086": "m",
"id08088": "m",
"id08089": "m",
"id08093": "f",
"id08094": "f",
"id08095": "m",
"id08096": "f",
"id08097": "f",
"id08098": "f",
"id08099": "f",
"id08106": "f",
"id08108": "m",
"id08110": "f",
"id08113": "f",
"id08114": "m",
"id08115": "f",
"id08119": "m",
"id08120": "f",
"id08121": "f",
"id08122": "f",
"id08123": "f",
"id08124": "f",
"id08125": "f",
"id08126": "f",
"id08127": "f",
"id08128": "f",
"id08129": "m",
"id08130": "f",
"id08132": "f",
"id08134": "f",
"id08137": "f",
"id08138": "f",
"id08139": "f",
"id08142": "f",
"id08143": "f",
"id08144": "f",
"id08145": "f",
"id08146": "f",
"id08147": "f",
"id08148": "m",
"id08149": "f",
"id08154": "f",
"id08155": "f",
"id08158": "f",
"id08159": "m",
"id08160": "m",
"id08162": "f",
"id08163": "f",
"id08165": "f",
"id08166": "f",
"id08167": "f",
"id08169": "f",
"id08170": "f",
"id08171": "f",
"id08172": "f",
"id08173": "f",
"id08175": "f",
"id08176": "f",
"id08178": "m",
"id08179": "m",
"id08180": "f",
"id08183": "m",
"id08184": "m",
"id08185": "m",
"id08186": "m",
"id08187": "f",
"id08188": "f",
"id08190": "f",
"id08191": "m",
"id08192": "m",
"id08193": "m",
"id08194": "m",
"id08195": "f",
"id08198": "m",
"id08200": "m",
"id08201": "m",
"id08202": "f",
"id08203": "f",
"id08204": "f",
"id08206": "m",
"id08209": "f",
"id08210": "m",
"id08211": "m",
"id08212": "m",
"id08213": "f",
"id08215": "f",
"id08217": "m",
"id08218": "m",
"id08219": "m",
"id08220": "m",
"id08221": "m",
"id08222": "f",
"id08230": "m",
"id08231": "m",
"id08232": "m",
"id08234": "m",
"id08235": "m",
"id08236": "f",
"id08238": "m",
"id08239": "m",
"id08240": "m",
"id08241": "m",
"id08242": "m",
"id08243": "m",
"id08244": "m",
"id08245": "m",
"id08246": "m",
"id08247": "m",
"id08249": "m",
"id08250": "m",
"id08251": "m",
"id08253": "m",
"id08258": "m",
"id08259": "m",
"id08260": "m",
"id08261": "m",
"id08263": "m",
"id08264": "m",
"id08266": "m",
"id08267": "m",
"id08271": "m",
"id08273": "m",
"id08274": "m",
"id08277": "m",
"id08278": "f",
"id08279": "m",
"id08280": "m",
"id08281": "f",
"id08282": "m",
"id08283": "m",
"id08284": "m",
"id08285": "m",
"id08286": "m",
"id08287": "m",
"id08288": "m",
"id08289": "m",
"id08290": "m",
"id08291": "m",
"id08292": "f",
"id08293": "f",
"id08294": "m",
"id08295": "m",
"id08297": "f",
"id08298": "f",
"id08299": "m",
"id08300": "f",
"id08302": "f",
"id08303": "f",
"id08305": "m",
"id08306": "f",
"id08307": "m",
"id08309": "m",
"id08311": "f",
"id08312": "m",
"id08313": "m",
"id08314": "m",
"id08315": "f",
"id08317": "f",
"id08318": "f",
"id08320": "f",
"id08321": "f",
"id08323": "m",
"id08324": "m",
"id08327": "f",
"id08329": "f",
"id08330": "m",
"id08331": "f",
"id08333": "f",
"id08334": "f",
"id08335": "f",
"id08337": "m",
"id08338": "m",
"id08339": "m",
"id08340": "f",
"id08343": "f",
"id08344": "f",
"id08347": "f",
"id08348": "m",
"id08349": "f",
"id08350": "f",
"id08351": "f",
"id08352": "f",
"id08353": "f",
"id08354": "f",
"id08355": "f",
"id08356": "f",
"id08358": "m",
"id08360": "m",
"id08362": "f",
"id08364": "m",
"id08365": "m",
"id08368": "f",
"id08370": "f",
"id08372": "m",
"id08373": "m",
"id08374": "m",
"id08375": "m",
"id08376": "m",
"id08377": "m",
"id08380": "m",
"id08381": "m",
"id08382": "f",
"id08392": "f",
"id08394": "f",
"id08395": "m",
"id08396": "f",
"id08397": "f",
"id08398": "f",
"id08399": "f",
"id08400": "f",
"id08402": "f",
"id08403": "f",
"id08404": "f",
"id08405": "f",
"id08406": "f",
"id08408": "f",
"id08411": "f",
"id08414": "f",
"id08415": "f",
"id08416": "f",
"id08417": "f",
"id08418": "m",
"id08419": "m",
"id08420": "m",
"id08421": "f",
"id08422": "f",
"id08423": "m",
"id08424": "m",
"id08426": "f",
"id08432": "m",
"id08433": "f",
"id08434": "m",
"id08437": "m",
"id08438": "m",
"id08440": "f",
"id08441": "f",
"id08442": "f",
"id08443": "m",
"id08445": "m",
"id08446": "m",
"id08451": "m",
"id08453": "m",
"id08454": "m",
"id08455": "m",
"id08456": "m",
"id08457": "m",
"id08458": "f",
"id08459": "f",
"id08462": "f",
"id08463": "f",
"id08464": "f",
"id08465": "f",
"id08467": "f",
"id08468": "f",
"id08470": "m",
"id08471": "f",
"id08472": "f",
"id08474": "f",
"id08475": "m",
"id08476": "m",
"id08477": "m",
"id08478": "m",
"id08480": "m",
"id08481": "f",
"id08482": "f",
"id08483": "m",
"id08489": "m",
"id08490": "m",
"id08492": "f",
"id08494": "m",
"id08496": "m",
"id08497": "m",
"id08498": "m",
"id08499": "m",
"id08500": "m",
"id08501": "m",
"id08502": "m",
"id08510": "m",
"id08511": "m",
"id08512": "m",
"id08513": "m",
"id08515": "m",
"id08516": "m",
"id08517": "m",
"id08518": "m",
"id08520": "m",
"id08521": "m",
"id08522": "m",
"id08523": "m",
"id08524": "m",
"id08525": "m",
"id08527": "m",
"id08528": "m",
"id08529": "m",
"id08530": "m",
"id08531": "m",
"id08532": "m",
"id08533": "m",
"id08534": "m",
"id08535": "m",
"id08536": "m",
"id08538": "m",
"id08540": "m",
"id08543": "f",
"id08544": "f",
"id08545": "f",
"id08546": "f",
"id08547": "f",
"id08548": "m",
"id08549": "f",
"id08551": "f",
"id08552": "m",
"id08553": "f",
"id08554": "m",
"id08555": "m",
"id08556": "m",
"id08563": "m",
"id08564": "m",
"id08565": "m",
"id08566": "m",
"id08567": "m",
"id08568": "m",
"id08569": "f",
"id08570": "f",
"id08572": "f",
"id08573": "f",
"id08574": "m",
"id08577": "f",
"id08578": "f",
"id08580": "m",
"id08581": "m",
"id08583": "m",
"id08584": "m",
"id08585": "m",
"id08586": "f",
"id08587": "m",
"id08588": "m",
"id08589": "m",
"id08590": "m",
"id08592": "m",
"id08593": "m",
"id08595": "f",
"id08597": "m",
"id08598": "m",
"id08599": "m",
"id08600": "m",
"id08601": "m",
"id08602": "m",
"id08603": "m",
"id08604": "m",
"id08605": "m",
"id08606": "m",
"id08607": "m",
"id08608": "m",
"id08609": "m",
"id08610": "m",
"id08611": "m",
"id08612": "m",
"id08613": "m",
"id08614": "m",
"id08615": "m",
"id08616": "m",
"id08617": "m",
"id08618": "m",
"id08622": "m",
"id08623": "m",
"id08624": "m",
"id08625": "m",
"id08626": "m",
"id08627": "m",
"id08628": "m",
"id08629": "m",
"id08633": "m",
"id08634": "m",
"id08635": "m",
"id08637": "f",
"id08638": "m",
"id08639": "f",
"id08640": "f",
"id08641": "f",
"id08642": "m",
"id08643": "m",
"id08645": "m",
"id08646": "m",
"id08647": "m",
"id08648": "m",
"id08649": "m",
"id08650": "m",
"id08651": "m",
"id08652": "m",
"id08653": "m",
"id08655": "m",
"id08656": "m",
"id08657": "m",
"id08658": "m",
"id08659": "m",
"id08660": "m",
"id08661": "m",
"id08663": "f",
"id08665": "f",
"id08666": "f",
"id08667": "m",
"id08668": "m",
"id08669": "m",
"id08670": "f",
"id08671": "f",
"id08672": "m",
"id08674": "m",
"id08676": "f",
"id08677": "m",
"id08678": "f",
"id08679": "f",
"id08680": "f",
"id08681": "m",
"id08683": "m",
"id08684": "m",
"id08685": "m",
"id08686": "m",
"id08687": "m",
"id08688": "m",
"id08689": "f",
"id08691": "m",
"id08692": "m",
"id08693": "m",
"id08694": "m",
"id08695": "m",
"id08696": "m",
"id08698": "f",
"id08699": "f",
"id08700": "f",
"id08701": "m",
"id08702": "f",
"id08703": "f",
"id08704": "f",
"id08705": "m",
"id08706": "m",
"id08707": "m",
"id08708": "m",
"id08709": "m",
"id08710": "m",
"id08711": "f",
"id08713": "m",
"id08714": "m",
"id08715": "f",
"id08716": "f",
"id08719": "m",
"id08721": "m",
"id08722": "m",
"id08723": "m",
"id08724": "m",
"id08725": "m",
"id08726": "m",
"id08727": "m",
"id08728": "m",
"id08730": "m",
"id08731": "m",
"id08732": "m",
"id08733": "m",
"id08736": "m",
"id08737": "f",
"id08738": "m",
"id08739": "m",
"id08740": "m",
"id08741": "m",
"id08742": "m",
"id08743": "m",
"id08745": "m",
"id08746": "m",
"id08747": "m",
"id08748": "m",
"id08749": "m",
"id08750": "m",
"id08751": "f",
"id08754": "m",
"id08755": "m",
"id08756": "f",
"id08757": "f",
"id08758": "m",
"id08760": "m",
"id08761": "f",
"id08762": "f",
"id08764": "f",
"id08765": "m",
"id08766": "f",
"id08768": "m",
"id08769": "m",
"id08771": "m",
"id08773": "m",
"id08774": "m",
"id08775": "f",
"id08776": "m",
"id08781": "m",
"id08782": "m",
"id08783": "f",
"id08784": "f",
"id08785": "f",
"id08786": "f",
"id08788": "f",
"id08789": "f",
"id08790": "f",
"id08792": "f",
"id08794": "f",
"id08795": "f",
"id08797": "f",
"id08799": "f",
"id08800": "f",
"id08801": "f",
"id08802": "f",
"id08803": "m",
"id08804": "m",
"id08805": "m",
"id08806": "f",
"id08808": "f",
"id08809": "f",
"id08810": "f",
"id08811": "f",
"id08819": "f",
"id08820": "f",
"id08821": "f",
"id08822": "m",
"id08823": "m",
"id08826": "f",
"id08830": "m",
"id08831": "m",
"id08832": "m",
"id08834": "f",
"id08837": "m",
"id08839": "f",
"id08840": "f",
"id08850": "f",
"id08853": "f",
"id08860": "m",
"id08862": "f",
"id08863": "f",
"id08864": "m",
"id08866": "m",
"id08868": "m",
"id08870": "f",
"id08871": "f",
"id08872": "f",
"id08873": "f",
"id08875": "f",
"id08876": "f",
"id08877": "f",
"id08878": "f",
"id08879": "f",
"id08881": "m",
"id08882": "m",
"id08883": "f",
"id08884": "m",
"id08885": "m",
"id08887": "m",
"id08890": "m",
"id08891": "m",
"id08892": "m",
"id08896": "f",
"id08898": "f",
"id08900": "f",
"id08901": "m",
"id08902": "m",
"id08903": "m",
"id08904": "m",
"id08906": "m",
"id08907": "m",
"id08909": "m",
"id08910": "m",
"id08911": "m",
"id08912": "m",
"id08913": "m",
"id08915": "m",
"id08916": "m",
"id08917": "f",
"id08919": "m",
"id08920": "m",
"id08923": "f",
"id08925": "m",
"id08926": "m",
"id08927": "m",
"id08928": "m",
"id08929": "m",
"id08930": "f",
"id08931": "m",
"id08933": "m",
"id08936": "f",
"id08937": "f",
"id08938": "m",
"id08940": "m",
"id08941": "f",
"id08942": "m",
"id08944": "m",
"id08945": "m",
"id08950": "m",
"id08953": "m",
"id08954": "m",
"id08955": "m",
"id08967": "m",
"id08968": "m",
"id08969": "m",
"id08971": "m",
"id08972": "m",
"id08973": "m",
"id08974": "m",
"id08975": "m",
"id08976": "m",
"id08977": "m",
"id08980": "m",
"id08981": "m",
"id08982": "m",
"id08983": "m",
"id08984": "m",
"id08992": "f",
"id08993": "f",
"id08994": "m",
"id08996": "m",
"id08997": "m",
"id08998": "m",
"id08999": "m",
"id09000": "m",
"id09003": "m",
"id09004": "m",
"id09005": "m",
"id09006": "m",
"id09007": "m",
"id09008": "m",
"id09009": "m",
"id09010": "f",
"id09012": "f",
"id09013": "m",
"id09015": "m",
"id09016": "m",
"id09017": "m",
"id09018": "m",
"id09020": "m",
"id09021": "m",
"id09022": "f",
"id09023": "m",
"id09024": "m",
"id09025": "m",
"id09027": "m",
"id09028": "m",
"id09029": "m",
"id09030": "m",
"id09031": "m",
"id09032": "m",
"id09033": "m",
"id09034": "m",
"id09035": "m",
"id09036": "m",
"id09038": "m",
"id09039": "m",
"id09040": "m",
"id09044": "m",
"id09046": "m",
"id09047": "m",
"id09048": "m",
"id09049": "m",
"id09051": "m",
"id09052": "m",
"id09053": "m",
"id09054": "f",
"id09055": "f",
"id09056": "f",
"id09057": "m",
"id09059": "m",
"id09061": "f",
"id09062": "m",
"id09063": "m",
"id09064": "m",
"id09065": "m",
"id09066": "f",
"id09067": "m",
"id09068": "m",
"id09069": "m",
"id09070": "m",
"id09071": "m",
"id09074": "f",
"id09075": "f",
"id09076": "f",
"id09078": "f",
"id09079": "m",
"id09081": "m",
"id09082": "f",
"id09083": "f",
"id09087": "f",
"id09088": "f",
"id09091": "f",
"id09092": "f",
"id09094": "m",
"id09095": "m",
"id09098": "f",
"id09099": "m",
"id09100": "m",
"id09108": "m",
"id09109": "m",
"id09110": "m",
"id09111": "m",
"id09112": "m",
"id09116": "f",
"id09119": "m",
"id09121": "m",
"id09123": "f",
"id09124": "m",
"id09125": "f",
"id09138": "m",
"id09139": "m",
"id09143": "m",
"id09144": "m",
"id09145": "f",
"id09146": "f",
"id09147": "f",
"id09149": "f",
"id09152": "m",
"id09153": "m",
"id09154": "f",
"id09155": "m",
"id09156": "m",
"id09159": "f",
"id09160": "f",
"id09161": "f",
"id09162": "f",
"id09163": "m",
"id09164": "m",
"id09171": "f",
"id09172": "f",
"id09174": "f",
"id09175": "f",
"id09179": "m",
"id09181": "f",
"id09185": "m",
"id09186": "m",
"id09187": "m",
"id09188": "f",
"id09189": "f",
"id09190": "f",
"id09192": "m",
"id09193": "m",
"id09194": "m",
"id09197": "f",
"id09198": "f",
"id09199": "f",
"id09200": "f",
"id09201": "f",
"id09202": "f",
"id09203": "f",
"id09204": "f",
"id09205": "f",
"id09207": "m",
"id09210": "m",
"id09211": "m",
"id09212": "m",
"id09213": "f",
"id09214": "m",
"id09215": "m",
"id09216": "f",
"id09217": "m",
"id09218": "m",
"id09220": "m",
"id09222": "m",
"id09224": "m",
"id09225": "m",
"id09226": "m",
"id09227": "m",
"id09229": "f",
"id09230": "m",
"id09231": "m",
"id09232": "f",
"id09234": "f",
"id09235": "m",
"id09236": "m",
"id09237": "m",
"id09238": "m",
"id09240": "f",
"id09242": "m",
"id09243": "m",
"id09245": "m",
"id09252": "m",
"id09255": "m",
"id09262": "m",
"id09263": "m",
"id09268": "m",
"id09269": "m",
"id09271": "m",
"id09272": "m"
}"""
GEN = {}
# FLAG_53 genders from Experiment/SCALE-CC-01/vox2_gender.json (embedded at build time; the Hugging Face vox2_meta.csv read
# failed in the smoke of 14/10 with KeyError 'VoxCeleb2 ID')
GEN.update({"vox2:" + s: g for s, g in json.loads(VOX2_GENDER_JSON).items()})
GEN.update({"vox2:" + s: g for s, g in zip(FK37.spk, FK37.gender)})
GEN.update({"vb:" + s: ("m" if str(g).lower().startswith("m") else "f") for s, g in zip(FKVB.spk, FKVB.gender)})

# -- frames: FLAG_37 rows in vox_frames_png.zip ("<row>.png"), FLAG_53 / VoxBlink in the FLAG_63 chunk zips ("spk/video/utt.png")
Z37 = one("vox_frames_png.zip") if FILES.get("vox_frames_png.zip") else one("vox_frames_png.zipdata")   # zipdata: the FLAG_87 bridge
ZCH = [p for p in FILES.get("frames_manifest.json", [])]
FR63 = ZCH[0].parent if ZCH else None
assert FR63 is not None, "attach the output of FLAG_63 (frames/)"
MEMBER = {}                                                     # member name -> zip path
with zipfile.ZipFile(Z37) as z:
    for n in z.namelist():
        MEMBER["v37/" + n] = Z37
for part in ("vox2b", "vb"):
    for zf in sorted([*(FR63 / part).glob("*.zip"), *(FR63 / part).glob("*.zipdata")]):
        with zipfile.ZipFile(zf) as z:
            for n in z.namelist():
                MEMBER[f"{part}/" + n] = zf
log(f"frames indexed: {sum(k.startswith('v37/') for k in MEMBER)} FLAG_37, {sum(k.startswith('vox2b/') for k in MEMBER)} FLAG_53, "
    f"{sum(k.startswith('vb/') for k in MEMBER)} VoxBlink")
_ZT = threading.local()


def frame_bytes(member):
    zs = getattr(_ZT, "z", None)
    if zs is None:
        zs = _ZT.z = {}
    zp = MEMBER[member]
    if zp not in zs:
        zs[zp] = zipfile.ZipFile(zp)
    return zs[zp].read(member.split("/", 1)[1])


# -- one table of faces and one of voices over the three sources; identity label: vox2:<spk> (FLAG_37 + FLAG_53) / vb:<spk>
faces, voices_ = [], []
for i, r in enumerate(FK37.itertuples()):
    faces.append(("v37", i, f"v37/{i}.png", "vox2:" + r.spk, r.video, r.spk not in drop42 | drop53))
for i, r in enumerate(FK2B.itertuples()):
    faces.append(("vox2b", i, f"vox2b/{r.spk}/{r.video}/{r.utt}.png", "vox2:" + r.spk, r.video, r.spk not in drop53))
for i, r in enumerate(FKVB.itertuples()):
    faces.append(("vb", i, f"vb/{r.spk}/{r.video}/{r.utt}.png", "vb:" + r.spk, r.video, r.spk not in dropvb))
FT = pd.DataFrame(faces, columns=["src", "row", "member", "id", "video", "ok"])
FT["ok"] &= FT.member.isin(set(MEMBER))
VE = np.concatenate([VE37, VE2B, VEVB])
VT = pd.DataFrame({"src": ["v37"] * len(VK37) + ["vox2b"] * len(VK2B) + ["vb"] * len(VKVB),
                   "id": list("vox2:" + VK37.spk) + list("vox2:" + VK2B.spk) + list("vb:" + VKVB.spk),
                   "video": list(VK37.video) + list(VK2B.video) + list(VKVB.video),
                   "ok": list(~VK37.spk.isin(drop42 | drop53)) + list(~VK2B.spk.isin(drop53)) + list(vb_voice_ok)})
fv = FT[FT.ok].groupby("id").indices; vv = VT[VT.ok].groupby("id").indices
fidx, vidx = FT[FT.ok].index.values, VT[VT.ok].index.values
FACE_OF = {s: fidx[ix] for s, ix in fv.items()}; VOICE_OF = {s: vidx[ix] for s, ix in vv.items()}
ids = sorted(s for s in set(FACE_OF) & set(VOICE_OF) if s in GEN)
rng = np.random.RandomState(CFG["seed"]); torch.manual_seed(CFG["seed"])
voxids = [s for s in ids if s.startswith("vox2:")]
held = sorted(rng.choice(voxids, min(CFG["held_out"], max(1, len(voxids) // 10)), replace=False)) if len(voxids) > 2 else []
POOL = {"vox": [s for s in voxids if s not in set(held)], "vb": [s for s in ids if s.startswith("vb:")]}
if SMOKE_IDS:
    POOL = {k: v[:SMOKE_IDS] for k, v in POOL.items()}; held = held[:2]
log(f"identities: VoxCeleb2 {len(POOL['vox'])} (+ {len(held)} held out), VoxBlink {len(POOL['vb'])} | faces {int(FT.ok.sum())}, "
    f"voices {int(VT.ok.sum())} | genders known {sum(s in GEN for s in ids)}/{len(ids)}")
assert POOL["vox"] and POOL["vb"]

## 5. CLIP-L + LoRA, exact fast preprocessing, wiring check

In [ ]:
print("[cell] 07 CLIP-L + LoRA (FACELORA-02s), fast exact preprocessing, wiri", flush=True)
# ============================================================================ CLIP-L + LoRA (FACELORA-02's), fast exact preprocessing, wiring check
from transformers import CLIPModel, CLIPImageProcessor
_proc = CLIPImageProcessor.from_pretrained(CLIPL_ID, revision=CLIPL_REV)
_cm = CLIPModel.from_pretrained(CLIPL_ID, revision=CLIPL_REV, torch_dtype=torch.float32)
VIS = nn.ModuleDict(dict(vm=_cm.vision_model, proj=_cm.visual_projection)).float()
del _cm
LAYERS, MLP_NAMES = VIS["vm"].encoder.layers, ("fc1", "fc2")
ENC_FWD = lambda x: VIS["proj"](VIS["vm"](pixel_values=x).pooler_output)          # = get_image_features
DIM = int(VIS["proj"].out_features)
for p in VIS.parameters():
    p.requires_grad = False
MEAN = torch.tensor(_proc.image_mean).view(3, 1, 1); STD = torch.tensor(_proc.image_std).view(3, 1, 1)


def PRE(im):
    return _proc(images=im.convert("RGB"), return_tensors="pt").pixel_values[0]


def PRE_FAST(im):
    """224 x 224 RGB frames: resize / centre crop are identities, so the processor reduces to rescale + normalise."""
    a = torch.from_numpy(np.asarray(im.convert("RGB"), dtype=np.float32) / 255.0).permute(2, 0, 1)
    return (a - MEAN) / STD


def load_frame(i, flip=False):
    im = Image.open(io.BytesIO(frame_bytes(FT.member.values[i]))).convert("RGB")
    if flip:
        im = im.transpose(Image.FLIP_LEFT_RIGHT)
    return PRE_FAST(im) if im.size == (224, 224) else PRE(im)


chk_rows = [int(i) for i in FT[FT.ok].groupby("src").head(16).index]
d = max(float((PRE_FAST(Image.open(io.BytesIO(frame_bytes(FT.member.values[i]))).convert("RGB"))
               - PRE(Image.open(io.BytesIO(frame_bytes(FT.member.values[i]))).convert("RGB"))).abs().max()) for i in chk_rows)
log(f"fast preprocessing vs CLIPImageProcessor on {len(chk_rows)} frames: max |diff| {d:.2e}")
assert d < 1e-4, "fast preprocessing differs from the processor"


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + F.linear(x, self.B @ self.A) * self.scale


class Ckpt(nn.Module):
    def __init__(self, blk):
        super().__init__(); self.blk = blk

    def forward(self, *a, **k):
        if self.training and torch.is_grad_enabled():
            return checkpoint(self.blk, *a, use_reentrant=False, **k)
        return self.blk(*a, **k)


for i in CFG["blocks"]:
    mlp = LAYERS[i].mlp
    for n in MLP_NAMES:
        setattr(mlp, n, LoRALinear(getattr(mlp, n), CFG["rank"], CFG["alpha"]))
if CFG.get("attn"):                                                       # FACELORA-05: + attention q / k / v / out
    for i in CFG["blocks"]:
        att = LAYERS[i].self_attn
        for n in ("q_proj", "k_proj", "v_proj", "out_proj"):
            assert isinstance(getattr(att, n), nn.Linear), (i, n, type(getattr(att, n)))
            setattr(att, n, LoRALinear(getattr(att, n), CFG["rank"], CFG["alpha"]))
for i in CFG["blocks"]:
    LAYERS[i] = Ckpt(LAYERS[i])
VIS = VIS.to(DEV)
VPROJ = nn.Linear(VE.shape[1], DIM).to(DEV)
with torch.no_grad():
    VPROJ.bias.zero_()
lora_params = [p for n, p in VIS.named_parameters() if p.requires_grad]
log(f"CLIP-L LoRA: {sum(p.numel() for p in lora_params) / 1e6:.2f} M trainable (+ vproj {sum(p.numel() for p in VPROJ.parameters()) / 1e6:.2f} M)")


@torch.no_grad()
def embed_rows(rows, bs=128):
    VIS.eval(); out = []
    with cf.ThreadPoolExecutor(8) as ex:
        for s in range(0, len(rows), bs):
            X = torch.stack(list(ex.map(load_frame, rows[s:s + bs]))).to(DEV)
            with torch.autocast("cuda", enabled=DEV.type == "cuda"):
                out.append(F.normalize(ENC_FWD(X).float(), dim=1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, DIM), np.float32)


STATE, CKPT = OUT / "facelora_state.pt", OUT / "facelora_ckpt.pt"
for n in ("facelora_state.pt", "facelora_ckpt.pt", "train_log.json"):         # resume: an earlier run of this notebook
    prev = [p for p in FILES.get(n, []) if p.parent.name == OUT.name
            or (n != "facelora_ckpt.pt" and re.search(r"feats[-_]ls[-_]c5s[-_]p", p.as_posix()))]   # B3-EVAL: saved adapter
    if prev and not (OUT / n).exists():
        shutil.copy(prev[0], OUT / n); log("resume:", n)
assert STATE.exists(), "B3-EVAL: attach quynhnhu82/flag2027-feats-ls-c5s-p (facelora_state.pt)"
REF = {"v37": D37 / "face_clipL.npy", "vox2b": D2B / "face_clipL.npy", "vb": DVB / "face_clipL.npy"}
WIRING = {}
if not STATE.exists() and not CKPT.exists():
    for src, f in REF.items():
        rows = [int(i) for i in FT[FT.ok & (FT.src == src)].index[:64]]
        ref = l2n_(np.load(f, mmap_mode="r")[FT.row.values[rows]].astype(np.float32))
        cos = (embed_rows(rows) * ref).sum(1)
        WIRING[src] = dict(rows=len(rows), mean_cos=round(float(cos.mean()), 5), min_cos=round(float(cos.min()), 5))
    log("wiring check at step 0 (adapter = frozen CLIP-L) vs the stored arrays:", WIRING)
    assert all(w["mean_cos"] >= 0.99 for w in WIRING.values()), f"the frames / encoder do not reproduce the stored CLIP-L: {WIRING}"

## 6. Training (registered settings; checkpoint every 500 steps)

In [ ]:
print("[cell] 08 training (registered: 16 VoxCeleb2 + 16 VoxBlink ids per ste", flush=True)
# ============================================================================ training (registered: 16 VoxCeleb2 + 16 VoxBlink ids per step)
fvid, vvid = FT.video.values, VT.video.values


def pick_ids(pool, n, balanced):
    if not balanced:
        return list(rng.choice(pool, min(n, len(pool)), replace=False))
    m = [s for s in pool if GEN[s] == "m"]; f = [s for s in pool if GEN[s] != "m"]
    h = n // 2
    return list(rng.choice(m, min(h, len(m)), replace=False)) + list(rng.choice(f, min(n - h, len(f)), replace=False))


_POOLG = {}


def sample_batch():
    P, K = CFG["P"], CFG["K"]
    spk = pick_ids(POOL["vox"], P // 2, CFG["sg"] > 0) + pick_ids(POOL["vb"], P - P // 2, CFG["sg"] > 0)
    fi, vi, lab, male = [], [], [], []
    for j, s in enumerate(spk):
        fr = FACE_OF[s]
        for _ in range(K):
            v = int(rng.choice(VOICE_OF[s]))
            cand = [i for i in fr if fvid[i] != vvid[v]] or list(fr)
            fi.append(int(rng.choice(cand))); vi.append(v); lab.append(j); male.append(GEN[s] == "m")
    flips = rng.rand(len(fi)) < CFG["flip"]
    with cf.ThreadPoolExecutor(8) as ex:
        X = torch.stack(list(ex.map(load_frame, fi, flips)))
    return X, torch.from_numpy(VE[vi]), torch.tensor(lab), torch.tensor(male)


def multi_pos_infonce(S, lab):
    same = (lab[:, None] == lab[None, :]).float()
    def one_(L):
        logp = L - torch.logsumexp(L, 1, keepdim=True)
        return -((logp * same).sum(1) / same.sum(1)).mean()
    return 0.5 * (one_(S) + one_(S.t()))


def sanity_eer():
    """Held-out VoxCeleb2 identities: face of one video vs voice of ANOTHER video; negatives = other held-out ids (all / same gender)."""
    from sklearn.metrics import roc_curve
    if len(held) < 2:
        return {}
    rows = [int(i) for s in held for i in FACE_OF[s][:6]]
    Fh = dict(zip(rows, embed_rows(rows)))
    VPROJ.eval()
    with torch.no_grad():
        vrows = [int(v) for s in held for v in VOICE_OF[s][:4]]
        Vh = dict(zip(vrows, F.normalize(VPROJ(torch.from_numpy(VE[vrows]).to(DEV)).float(), dim=1).cpu().numpy()))
    r2 = np.random.RandomState(1); pos, neg, negg = [], [], []
    for s in held:
        fr = [i for i in FACE_OF[s][:6]]
        for v in VOICE_OF[s][:4]:
            cand = [i for i in fr if fvid[i] != vvid[v]]
            if cand:
                pos.append(float(Fh[int(r2.choice(cand))] @ Vh[int(v)]))
            o = held[r2.randint(len(held))]
            if o != s:
                sc = float(Fh[int(r2.choice(FACE_OF[o][:6]))] @ Vh[int(v)]); neg.append(sc)
                if GEN[o] == GEN[s]:
                    negg.append(sc)
    def eer(p, n):
        if len(p) < 2 or len(n) < 2:
            return None
        y = np.r_[np.ones(len(p)), np.zeros(len(n))]; s_ = np.r_[p, n]
        fpr, tpr, _ = roc_curve(y, s_); fnr = 1 - tpr; i = np.nanargmin(np.abs(fnr - fpr)); return round(100 * float(fpr[i] + fnr[i]) / 2, 2)
    return {"ng": eer(pos, neg), "g": eer(pos, negg) if len(negg) > 20 else None, "pos": len(pos), "neg": len(neg)}


opt = torch.optim.AdamW([{"params": lora_params, "lr": CFG["lr_lora"]}, {"params": VPROJ.parameters(), "lr": CFG["lr_proj"]}], weight_decay=CFG["wd"])
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / CFG["warmup"]) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / max(1, CFG["steps"])))))
scaler = torch.amp.GradScaler(enabled=DEV.type == "cuda")
LOG = {"cfg": CFG, "wiring": WIRING, "loss": [], "sanity": {}, "pools": {k: len(v) for k, v in POOL.items()}}
start = 1
if STATE.exists():
    st = torch.load(STATE, map_location="cpu", weights_only=False)
    VIS.load_state_dict({**VIS.state_dict(), **st["vis"]}); VPROJ.load_state_dict(st["vproj"])
    LOG = json.loads((OUT / "train_log.json").read_text()) if (OUT / "train_log.json").exists() else LOG
    start = CFG["steps"] + 1; log("adapter loaded (training done earlier):", st["cfg"]["arm"])
elif CKPT.exists():
    ck = torch.load(CKPT, map_location="cpu", weights_only=False)
    VIS.load_state_dict({**VIS.state_dict(), **ck["vis"]}); VPROJ.load_state_dict(ck["vproj"])
    opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    rng.set_state(ck["rng"]); torch.set_rng_state(ck["torch_rng"]); LOG = ck["log"]; start = ck["step"] + 1
    log(f"resume from checkpoint at step {ck['step']}")
if start == 1:
    LOG["sanity"]["0"] = sanity_eer(); log("sanity EER before training:", LOG["sanity"]["0"])
t0 = time.time()
BUDGET_H = float(os.environ.get("LORA_BUDGET_H", "0") or 0)          # 14/10: stop cleanly (checkpoint) after this many training hours
stopped = False
for step in range(start, CFG["steps"] + 1):
    VIS.train(); VPROJ.train()
    X, V_, lab, male = sample_batch()
    X, V_, lab, male = X.to(DEV), V_.to(DEV), lab.to(DEV), male.to(DEV)
    with torch.autocast("cuda", enabled=DEV.type == "cuda"):
        f = F.normalize(ENC_FWD(X).float(), dim=1); v = F.normalize(VPROJ(V_).float(), dim=1)
        SIM = f @ v.t() / CFG["tau"]                                  # not S: S is the data cell's dict of organiser lists
        loss = multi_pos_infonce(SIM, lab)
        if CFG["sg"] > 0:                                              # B3: same-gender negatives only (g cells)
            for sel in (male, ~male):
                if sel.sum() > 2:
                    loss = loss + CFG["sg"] * multi_pos_infonce(SIM[sel][:, sel], lab[sel])
    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    LOG["loss"].append(float(loss))
    if step % 50 == 0 or step == CFG["steps"]:
        log(f"step {step}/{CFG['steps']} loss {np.mean(LOG['loss'][-50:]):.4f} [{time.time() - t0:.0f}s]"
            + (f" | GPU peak {torch.cuda.max_memory_allocated() / 2 ** 30:.1f} GB" if DEV.type == "cuda" else ""))
    if step in (1000, CFG["steps"] // 2, CFG["steps"]):
        LOG["sanity"][str(step)] = sanity_eer(); log(f"sanity EER at step {step}:", LOG["sanity"][str(step)])
    if step % CFG["ckpt_every"] == 0 and step < CFG["steps"]:
        torch.save(dict(vis={k: v.cpu() for k, v in VIS.state_dict().items() if k.endswith((".A", ".B"))}, vproj=VPROJ.state_dict(),
                        opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(), rng=rng.get_state(),
                        torch_rng=torch.get_rng_state(), log=LOG, step=step), CKPT)
        log(f"checkpoint at step {step}")
    if BUDGET_H and (time.time() - t0) / 3600 > BUDGET_H and step < CFG["steps"]:
        torch.save(dict(vis={k: v.cpu() for k, v in VIS.state_dict().items() if k.endswith((".A", ".B"))}, vproj=VPROJ.state_dict(),
                        opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(), rng=rng.get_state(),
                        torch_rng=torch.get_rng_state(), log=LOG, step=step), CKPT)
        log(f"time budget {BUDGET_H} h reached: checkpoint at step {step}/{CFG['steps']}; attach this output and run again to resume")
        stopped = True
        break
if start <= CFG["steps"] and not stopped:
    torch.save({"vis": {k: v.cpu() for k, v in VIS.state_dict().items() if k.endswith((".A", ".B"))}, "vproj": VPROJ.state_dict(),
                "cfg": CFG, "wiring": WIRING}, STATE)
    (OUT / "train_log.json").write_text(json.dumps(LOG, indent=1))
    log("adapter saved ->", STATE.name)
VIS.eval(); VPROJ.eval()
if stopped:                                                     # partial training: no adapted array is extracted in this run
    ENCODERS, EXT_ENCODERS = [], []
    log("partial run: the extraction cells below do nothing; the checkpoint is in the output")

## 7. The adapted encoder

In [ ]:
print("[cell] 09 the adapted encoder in the FLAG_12 encoder interface (MAV-Ce", flush=True)
# ============================================================================ the adapted encoder in the FLAG_12 encoder interface (MAV-Celeb rows)
class Adapted:
    kind = "face"

    def __init__(self):
        VIS.eval()

    def prep(self, im):
        return PRE(im)

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        with torch.autocast("cuda", enabled=use_amp and DEV.type == "cuda"):
            return {"": ENC_FWD(x.to(DEV)).float().cpu().numpy()}


NEW = {ARR + "V": Adapted}


def build(name):
    return NEW[name]()


def release(keep_ib=False):
    import gc
    gc.collect()


ARRAYS_NEW = {ARR + "V": ("face", {"": ARR})}
MAIN_NEW = {k: "" for k in NEW}
RESHAPE = {}

## 7b. Saved train / v1 / v2 arrays copied in: only the test lists are embedded

In [ ]:
print("[cell] 10 CLEANTUNE-01 Evaluation path (2/11): every file of the SAVED", flush=True)
# CLEANTUNE-01 Evaluation path (2/11): every file of the SAVED clean dataset except the test lists is copied in (train / v1 / v2 /
# VoxCeleb2 / VoxBlink arrays, adapter, bn42_clean.json), so this output alone replaces the dataset in the CC-head run (FLAG_112x);
# only the test lists attached (dev = R11 rehearsal, Evaluation on 11/11) are embedded here through the saved adapter
ref_dir = next((p.parent for p in FILES.get("face_clipC5_train.npy", []) if OUT.name not in p.as_posix()), None)
assert ref_dir is not None and (ref_dir / "bn42_clean.json").exists(), "attach the saved clean adapter dataset (bn42_clean.json)"
for f in sorted(ref_dir.rglob("*")):
    d = OUT / f.relative_to(ref_dir)
    if f.is_file() and not re.match(r"face_clipC5_(no_)?gender_", f.name) and not d.exists():
        d.parent.mkdir(parents=True, exist_ok=True); shutil.copy(f, d); log("saved file copied in:", f.relative_to(ref_dir))

## 8. `clipC5` of every MAV-Celeb row (FLAG_12 loop, resumable)

In [ ]:
print("[cell] 11 running", flush=True)
# ============================================================================ running
def run_faces_tensor(enc, srcs, bs=64, label=""):
    out, t0 = {}, time.time()
    for s in range(0, len(srcs), bs):
        x = torch.stack([enc.prep(read_rgb(p() if callable(p) else p)) for p in srcs[s:s + bs]])
        for k, v in enc(x).items():
            out.setdefault(k, []).append(v)
        if (s // bs) % 40 == 0:
            log(f"  {label} {s + len(x)}/{len(srcs)}  {time.time() - t0:.0f}s")
    return {k: np.concatenate(v).astype(np.float32) for k, v in out.items()}


def run_voices_single(enc, n, load, label=""):
    """One file at a time: no padding enters a mean."""
    out, t0 = {}, time.time()
    for i in range(n):
        for k, v in enc(load(i)).items():
            out.setdefault(k, []).append(np.atleast_1d(v))
        if i % 1000 == 0 or i == n - 1:
            el = time.time() - t0
            log(f"  {label} {i + 1}/{n}  {el:.0f}s  eta {el / (i + 1) * (n - i - 1):.0f}s")
    return {k: np.stack(v).astype(np.float32) for k, v in out.items()}


# output names: {kind}_{array}_{split}.npy for organiser splits, ext_{src}_{kind}_{array}.npy for v1/v2
ARRAYS = {"arcface": ("face", {"": "arcface", "_detok": "arcface_detok"}),
          "ecapa": ("voice", {"192": "ecapa192", "6144": "ecapa6144"}),
          "vitage": ("face", {"": "vitage", "prob": "vitageprob"}),
          "w2v2ag": ("voice", {"": "w2v2ag", "age": "w2v2agage", "gen": "w2v2aggen"}),
          "ibv": ("face", {"": "ibv"}), "iba": ("voice", {"": "iba"})}
MAIN = {"arcface": "", "ecapa": "192", "vitage": "", "w2v2ag": "", "ibv": "", "iba": ""}
ARRAYS.update(ARRAYS_NEW); MAIN.update(MAIN_NEW)                         # this notebook's arrays


def _dt(v, split):
    """float16 for the v1/v2 rows (as FLAG_12) and for every wide array (layers / tokens); float32 otherwise."""
    return v if (split and v.ndim == 2 and v.shape[1] <= 4096) or v.ndim == 1 else v.astype(np.float16)


def target(name, suffix, split=None, src=None):
    kind, names = ARRAYS[name]
    return OUT / (f"{kind}_{names[suffix]}_{tag(split)}.npy" if split else f"ext_{src}_{kind}_{names[suffix]}.npy")


def complete(name, split=None, src=None):
    return all(target(name, s, split, src).exists() for s in ARRAYS[name][1])


def save(name, R, split=None, src=None):
    for s, v in R.items():                         # main array last: its presence marks the job as done
        if s != MAIN[name]:
            np.save(target(name, s, split, src), _dt(v, split))
    np.save(target(name, MAIN[name], split, src), _dt(R[MAIN[name]], split))


def extract(name, enc, split=None, src=None):
    kind = ARRAYS[name][0]
    if split:
        sp = S[split]
        faces = [sp["root"] / p for p in sp["jpg"]]
        load = lambda i: load_audio(sp["root"] / sp["wav"][i])
        n, durs, label = len(sp["wav"]), sp["dur"], split
    else:
        rows = EXT[src]
        faces = [rows.read(p) for p in rows.face] if name == "arcface" else [(lambda p=p: rows.read(p)) for p in rows.face]
        load = lambda i: centre_crop(load_audio(io.BytesIO(rows.read(rows.voice[i]))), rows.L)
        n, label = len(rows.voice), src
        durs = np.full(n, 12.0, np.float32)          # upper bound after the crop: batches stay within budget
    if kind == "face":
        R = enc.embed(faces, label=label) if name == "arcface" else run_faces_tensor(enc, faces, label=f"{name} {label}")
    elif name == "ecapa":
        R = enc.embed(load, n, durs, label=label)
    else:
        R = run_voices_single(enc, n, load, label=f"{name} {label}")
    save(name, R, split, src)
    log(f"saved {target(name, MAIN[name], split, src).name} {R[MAIN[name]].shape}")


def self_check(name, enc):
    """Batching / fp16 must not change the embeddings (min cosine over a few rows; must be > 0.995)."""
    k = next(k for k in S if k != "train") if len(S) > 1 else next(iter(S))
    sp = S[k]
    if name == "arcface":
        return {}
    if name == "ecapa":
        idx = list(np.argsort(-sp["dur"])[:3]) + list(range(min(5, len(sp["wav"]))))
        sigs = [load_audio(sp["root"] / sp["wav"][i]) for i in idx]
        b, s = enc.batch(sigs), [enc.batch([x])[0] for x in sigs]
        return {"batch192": float(cos_rows([r[0] for r in b], [r[0] for r in s]).min()),
                "batch6144": float(cos_rows([r[1] for r in b], [r[1] for r in s]).min())}
    if ARRAYS[name][0] == "face":
        x = torch.stack([enc.prep(read_rgb(sp["root"] / p)) for p in sp["jpg"][:6]])
        full = enc(x, use_amp=False)[""]
        single = np.concatenate([enc(x[i:i + 1], use_amp=False)[""] for i in range(len(x))])
        return {"batch": float(cos_rows(full, single).min()), "fp16": float(cos_rows(full, enc(x)[""]).min())}
    waves = [load_audio(sp["root"] / p) for p in sp["wav"][:3]]
    return {"fp16": float(cos_rows([enc(w, use_amp=False)[""] for w in waves], [enc(w)[""] for w in waves]).min())}


CHECKS, FAILED = {}, {}
for name in ENCODERS:
    jobs = [("split", k) for k in S if not complete(name, split=k)]
    if name in EXT_ENCODERS:
        jobs += [("src", s) for s in EXT if not complete(name, src=s)]
    if not jobs:
        log(f"== {name}: every array already exists -> skipped")
        continue
    t0 = time.time()
    try:
        enc = build(name)
        CHECKS[name] = self_check(name, enc)
        log(name, "self-check:", CHECKS[name])
        assert not CHECKS[name] or min(CHECKS[name].values()) > 0.995, f"{name}: batching / fp16 changes the output"
        for what, key in jobs:
            try:
                extract(name, enc, **({"split": key} if what == "split" else {"src": key}))
            except Exception as e:                  # one bad source must not cost the other splits
                FAILED[f"{name}:{key}"] = f"{type(e).__name__}: {e}"
                log(f"!! {name} on {key} FAILED: {FAILED[f'{name}:{key}']}")
    except Exception as e:
        FAILED[name] = f"{type(e).__name__}: {e}"
        log(f"!! {name} FAILED -> skipped: {FAILED[name]}")
    enc = None
    release(keep_ib=(name in ('ibv',)))
    log(f"== {name} done in {time.time() - t0:.0f}s")
print("failed:", FAILED or "none")

## 9. Check: re-extracted arrays vs the saved B3 arrays

In [ ]:
print("[cell] 12 B3-EVAL: re-extracted arrays vs the saved ones (no VoxCeleb2", flush=True)
# ============================================================================ B3-EVAL: re-extracted arrays vs the saved ones (no VoxCeleb2 / VoxBlink rows)
ref_dir = next((p.parent for p in FILES.get("face_clipC5_train.npy", []) if OUT.name not in p.as_posix()), None)
assert ref_dir is not None, "attach quynhnhu82/flag2027-feats-ls-c5s-p"
chk = {}
for f in sorted(OUT.glob("face_clipC5_*.npy")) + sorted(OUT.glob("ext_*_face_clipC5.npy")):
    r = ref_dir / f.name
    if ONLY_UNDER and not (f.name.startswith("ext_") or f.name.endswith("_train.npy")):   # Evaluation lists (11/11)
        chk[f.name] = "Evaluation list: no saved reference"; continue
    if not r.exists():
        chk[f.name] = "no reference (new list)"; continue
    a, b = np.load(f).astype(np.float32), np.load(r).astype(np.float32)
    n = min(len(a), len(b)) if SMOKE else len(b)
    assert len(a) == n or SMOKE, (f.name, a.shape, b.shape)
    a, b = a[:n], b[:n]
    ok = np.isfinite(a).all(1) & np.isfinite(b).all(1)
    cos = (l2n_(a[ok]) * l2n_(b[ok])).sum(1)
    chk[f.name] = dict(rows=int(n), finite=int(ok.sum()), mean_cos=round(float(cos.mean()), 6), min_cos=round(float(cos.min()), 6))
log("B3-EVAL re-extracted vs saved:", json.dumps(chk, indent=1))
bad = {k: v for k, v in chk.items() if isinstance(v, dict) and v["min_cos"] < 0.999}
assert not bad, f"the saved adapter does not reproduce the saved arrays: {bad}"
log("OK: attach this output (feats_clipb3_x/) to FLAG_82b; on Evaluation set E2E_SPLIT=eval there")

## 10. Manifest

In [ ]:
print("[cell] 13 man = created: time.strftime(Y-m-d H:M), splits: k: len(vwav", flush=True)
man = {"created": time.strftime("%Y-%m-%d %H:%M"), "splits": {k: len(v["wav"]) for k, v in S.items()},
       "external": {k: [len(r.face), len(r.voice)] for k, r in EXT.items()}, "encoders": ENCODERS,
       "checks": CHECKS, "failed": FAILED, "audio": AUDIO_STATS, "broken": BROKEN, "resumed_arrays": len(SEEDED),
       "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
       "files": {f.name: list(np.load(f, mmap_mode="r").shape) for f in sorted(OUT.glob("*.npy"))}}
try:
    import speechbrain, insightface
    man["versions"].update(speechbrain=speechbrain.__version__, insightface=insightface.__version__)
except Exception:
    pass
man["reshape"] = RESHAPE                 # flattened wide arrays: row -> (L, D)
(OUT / "feats_manifest.json").write_text(json.dumps(man, indent=1))
shutil.rmtree(WORKDIR / "raw", ignore_errors=True)
tot = sum(p.stat().st_size for p in OUT.glob("*")) / 1e9
print(f"{len(man['files'])} arrays, {tot:.2f} GB in {OUT} | failed: {FAILED or 'none'}")
print("NEXT: Save Version (Save & Run All) -> Output -> New Dataset -> name it flag2027-feats-clipc5c-test-p (re-run with this output attached until every encoder is done)")